In [1]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Mounted at /content/drive
GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
%%capture
!pip uninstall -y transformers # Uninstall existing transformers
!rm -rf "/content/cache" # Remove the cache directory
!pip install -q transformers==4.40.0 datasets==2.19.0 peft==0.10.0 \
               accelerate==0.29.3 evaluate==0.4.1 jiwer==3.0.3 \
               soundfile librosa openpyxl audioread ffmpeg-python
!apt-get install -qq ffmpeg
print("Dependencies installed")

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("itssabtain/pashto-asr-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'pashto-asr-dataset' dataset.
Path to dataset files: /kaggle/input/pashto-asr-dataset


In [ ]:
import os
import pandas as pd
import glob
import re # Import re for sorted_audio_files

# The path to the downloaded Kaggle dataset is available as 'path' from the previous cell
kaggle_dataset_base_path = path

print(f"Loading Kaggle dataset for preprocessing from: {kaggle_dataset_base_path}")

# --- Helper functions (copied from XHai6zY9z9AM to ensure availability) ---
AUDIO_EXTS = {".wav", ".mp3", ".ogg", ".mp4", ".m4a", ".flac"}

def find_excel(folder):
    """Return the first .xlsx/.xls file found in folder."""
    for ext in ("*.xlsx", "*.xls"):
        hits = glob.glob(os.path.join(folder, ext))
        if hits:
            return hits[0]
    return None

def load_transcriptions(excel_path):
    """Load single-column Excel → list of strings ('' for empty rows)."""
    df = pd.read_excel(excel_path, header=None, dtype=str)
    # Flatten to a list; NaN → empty string
    return df.iloc[:, 0].fillna("").str.strip().tolist()

def sorted_audio_files(folder):
    """Return audio files sorted by name (natural sort)."""
    files = []
    for f in os.listdir(folder):
        if os.path.splitext(f)[1].lower() in AUDIO_EXTS:
            files.append(os.path.join(folder, f))
    # natural sort by numeric parts in filename
    files.sort(key=lambda p: [int(c) if c.isdigit() else c
                               for c in re.split(r'(\d+)', os.path.basename(p))])
    return files
# --- End Helper functions ---

# The actual dataset content is located under a 'dataset' subfolder
kaggle_dataset_actual_root = os.path.join(kaggle_dataset_base_path, "dataset")

print(f"\nContents of {kaggle_dataset_base_path}:")
!ls -R "{kaggle_dataset_base_path}"

if not os.path.isdir(kaggle_dataset_actual_root):
    raise FileNotFoundError(f"Expected 'dataset' directory not found at: {kaggle_dataset_actual_root}")

kaggle_raw_pairs = []
skipped_kaggle_count = 0

# Iterate through subdirectories (Agriculture, Food_Services, etc.)
for category_folder_name in os.listdir(kaggle_dataset_actual_root):
    category_folder_path = os.path.join(kaggle_dataset_actual_root, category_folder_name)

    if os.path.isdir(category_folder_path):
        # Find the Excel transcription file in the category folder
        excel_path = find_excel(category_folder_path)
        if excel_path is None:
            print(f"  No Excel transcription file found in category: {category_folder_name}. Skipping.")
            continue

        # Find the 'audios' subfolder
        audio_folder = os.path.join(category_folder_path, "audios")
        if not os.path.isdir(audio_folder):
            print(f"  'audios' folder not found in category: {category_folder_name}. Skipping.")
            continue

        print(f"  Processing category: {category_folder_name}")
        print(f"    Transcription Excel at: {excel_path}")
        print(f"    Audio files in: {audio_folder}")

        transcripts = load_transcriptions(excel_path)
        audio_files = sorted_audio_files(audio_folder)

        # Pair transcriptions with audio files. The 'file' column in the CSV is not used here,
        # instead we assume a 1:1 mapping based on sorted order, similar to Drive data processing.
        paired_in_category = 0
        for audio_filename, transcription_text in zip(audio_files, transcripts):
            full_audio_path = audio_filename # audio_files already contains full paths

            # Filter out empty or missing transcriptions and missing audio files
            if pd.isna(transcription_text) or str(transcription_text).strip() == "":
                skipped_kaggle_count += 1
                continue
            if not os.path.exists(full_audio_path):
                print(f"      Warning: Audio file not found for '{os.path.basename(full_audio_path)}' at '{full_audio_path}'. Skipping.")
                skipped_kaggle_count += 1
                continue

            kaggle_raw_pairs.append({"audio": full_audio_path, "text": str(transcription_text).strip()})
            paired_in_category += 1
        print(f"    Paired {paired_in_category} samples in {category_folder_name}")

# Overwrite the 'raw_pairs' variable that was created from the Drive dataset
# This ensures subsequent steps (validation, feature extraction) use the Kaggle data.
raw_pairs = kaggle_raw_pairs

print(f"\nKaggle dataset preprocessing complete.")
print(f"  Total usable pairs from Kaggle: {len(raw_pairs)}")
print(f"  Skipped samples (empty transcription or missing audio): {skipped_kaggle_count}")
print("The 'raw_pairs' variable has been updated to use the Kaggle dataset. You can now proceed with the next data processing steps.")

Loading Kaggle dataset for preprocessing from: /kaggle/input/pashto-asr-dataset

Contents of /kaggle/input/pashto-asr-dataset:
/kaggle/input/pashto-asr-dataset:
dataset

/kaggle/input/pashto-asr-dataset/dataset:
Agriculture  Food_Services  General  Health1  Health2  Services

/kaggle/input/pashto-asr-dataset/dataset/Agriculture:
Agriculture_Transcript.xlsx  audios

/kaggle/input/pashto-asr-dataset/dataset/Agriculture/audios:
100.ogg  140.ogg  180.ogg  21.ogg   25.ogg   29.ogg   339.ogg  379.ogg	66.ogg
101.ogg  141.ogg  181.ogg  220.ogg  260.ogg  2.ogg    33.ogg   37.ogg	67.ogg
102.ogg  142.ogg  182.ogg  221.ogg  261.ogg  300.ogg  340.ogg  380.ogg	68.ogg
103.ogg  143.ogg  183.ogg  222.ogg  262.ogg  301.ogg  341.ogg  381.ogg	69.ogg
104.ogg  144.ogg  184.ogg  223.ogg  263.ogg  302.ogg  342.ogg  382.ogg	6.ogg
105.ogg  145.ogg  185.ogg  224.ogg  264.ogg  303.ogg  343.ogg  383.ogg	70.ogg
106.ogg  146.ogg  186.ogg  225.ogg  265.ogg  304.ogg  344.ogg  384.ogg	71.ogg
107.ogg  147.ogg  187.ogg  

In [ ]:
import os

# ── Paths ─────────────────────────────────────────────────────────────────────
# DRIVE_ROOT was previously used for Google Drive datasets, but data is now
# sourced from Kaggle. OUTPUT_DIR and CACHE_DIR are for model artifacts.
OUTPUT_DIR = "/content/drive/MyDrive/Audios_Phusto/whisper-pashto-lora"
CACHE_DIR  = "/content/cache"

# NOTE: os.makedirs for OUTPUT_DIR and CACHE_DIR will be handled implicitly
# when those paths are first used, or in dedicated setup cells, not here.

# ── Dataset Folders ──────────────────────────────────────────────────────────
# DATASET_FOLDERS was previously used for Google Drive datasets. With the
# Kaggle dataset, the structure is handled directly during loading.
# DATASET_FOLDERS = [
#     "Agriculture",
#     "General",
#     "Food+Services",
#     "Health1",
#     "Health2",
#     "Services",
# ]

# ── Training hyper-params (memory-safe for T4 16 GB) ─────────────────────────
MODEL_NAME       = "openai/whisper-small"
LANGUAGE         = "Pashto"
TASK             = "transcribe"
SAMPLE_RATE      = 16_000
MAX_AUDIO_SEC    = 30        # skip clips longer than this
BATCH_SIZE       = 4         # reduce to 2 if you hit OOM
GRAD_ACCUM       = 4         # effective batch = 16
LEARNING_RATE    = 2e-4      # lower LR = more stable for low-resource Pashto
NUM_EPOCHS       = 5
WARMUP_STEPS     = 200       # longer warmup matches lower LR
EVAL_STEPS       = 200
SAVE_STEPS       = 200
MAX_STEPS        = 10000     # more steps per round for harder language (increased)
TRAIN_SPLIT      = 0.9       # 90% train, 10% eval
FP16             = True      # keep True for T4

# ── LoRA config ───────────────────────────────────────────────────────────────
LORA_R           = 128       # higher rank = more adaptation capacity (increased)
LORA_ALPHA       = 256       # keep alpha = 2x rank (adjusted for LORA_R)
LORA_DROPOUT     = 0.15      # slightly higher dropout for 5k dataset size (increased)

print(" Config ready")
# Drive root is no longer directly used for dataset loading
print(f"   Output dir : {OUTPUT_DIR}")

 Config ready
   Output dir : /content/drive/MyDrive/Audios_Phusto/whisper-pashto-lora


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# Step 4 — Parallel validation (metadata only, no audio decoded)
# Stores ONLY file paths + transcriptions in an Arrow dataset.
# NO HuggingFace Audio cast here — we decode manually in Step 5.
# ─────────────────────────────────────────────────────────────────────────
import os, gc, json as _json, subprocess, psutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from datasets import Dataset

ARROW_CACHE = "/content/hf_cache"
os.makedirs(ARROW_CACHE, exist_ok=True)

def get_duration_ffprobe(path):
    """Duration in seconds via ffprobe (handles ALL formats). -1 on failure."""
    try:
        cmd = ["ffprobe", "-v", "error",
               "-show_entries", "format=duration",
               "-of", "json", path]
        out = subprocess.run(cmd, capture_output=True, text=True, timeout=15)
        data = _json.loads(out.stdout)
        return float(data["format"]["duration"])
    except Exception:
        return -1

def validate_one(pair):
    path, text = pair["audio"], pair["text"]
    if not os.path.isfile(path):
        return None, f"missing: {path}"
    dur = get_duration_ffprobe(path)
    if dur <= 0:
        return None, f"unreadable: {os.path.basename(path)}"
    if dur > MAX_AUDIO_SEC:
        return None, f"too long ({dur:.1f}s): {os.path.basename(path)}"
    return (path, text), None

print(f"Validating {len(raw_pairs)} files in parallel (ffprobe, 4 threads)...")
print(f"RAM before: {psutil.virtual_memory().used/1e9:.1f} GB")

valid_paths, valid_texts, skip_log = [], [], []

with ThreadPoolExecutor(max_workers=4) as pool:
    futures = {pool.submit(validate_one, p): i for i, p in enumerate(raw_pairs)}
    done = 0
    for fut in as_completed(futures):
        result, err = fut.result()
        done += 1
        if done % 250 == 0 or done == len(raw_pairs):
            print(f"  {done:>5}/{len(raw_pairs)}  RAM: {psutil.virtual_memory().used/1e9:.1f} GB", end="\r")
        if result is None:
            skip_log.append(err)
        else:
            valid_paths.append(result[0])
            valid_texts.append(result[1])

print(f"\n Valid   : {len(valid_paths)}")
print(f"   Skipped : {len(skip_log)}")
if skip_log[:5]: print("   Examples:", skip_log[:5])

# ── Arrow dataset stores PATHS as plain strings — no Audio cast ──────────────
# We decode with librosa in Step 5, which handles every format via ffmpeg.
raw_dataset = Dataset.from_dict({"path": valid_paths, "sentence": valid_texts})
split    = raw_dataset.train_test_split(test_size=1 - TRAIN_SPLIT, seed=42)
train_ds = split["train"]
eval_ds  = split["test"]

del valid_paths, valid_texts, raw_pairs
gc.collect()
print(f"   Train: {len(train_ds)}  |  Eval: {len(eval_ds)}")
print(f"RAM after : {psutil.virtual_memory().used/1e9:.1f} GB  ← no audio in RAM")
print(" Step 4 complete")

Validating 5098 files in parallel (ffprobe, 4 threads)...
RAM before: 1.7 GB
   5098/5098  RAM: 2.1 GB
 Valid   : 5081
   Skipped : 17
   Examples: ['too long (33.4s): 30.wav', 'too long (31.9s): 200.wav', 'too long (32.3s): 355.wav', 'too long (30.8s): 452.wav', 'too long (34.5s): 672.wav']
   Train: 4572  |  Eval: 509
RAM after : 2.1 GB  ← no audio in RAM
 Step 4 complete


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# Step 5 — Feature Extraction  (parallel, memory-safe, ALL formats)
#
# Root-cause fix: soundfile (used by HF Audio cast) crashes on .mp4/.m4a/.ogg
# Solution: decode every file with librosa which shells out to ffmpeg —
#           handles wav, mp3, ogg, mp4, m4a, flac, anything ffmpeg knows.
#
# Memory strategy:
#   batched=False  → 1 sample decoded at a time, freed after each call
#   num_proc=2     → 2 CPU workers overlap Drive I/O with mel computation
#   writer_batch_size=50 → Arrow flushes to SSD every 50 rows
# ─────────────────────────────────────────────────────────────────────────
import gc, os, psutil, warnings
import numpy as np
import torch
import librosa
from transformers import WhisperProcessor

warnings.filterwarnings("ignore")  # suppress librosa UserWarnings

processor = WhisperProcessor.from_pretrained(
    MODEL_NAME, language=LANGUAGE, task=TASK, cache_dir=CACHE_DIR
)

NUM_MAP_WORKERS   = 2    # Colab free = 2 vCPUs
WRITER_BATCH_SIZE = 50   # flush Arrow buffer every 50 rows

def prepare_features(batch):
    """
    Decodes audio with librosa+ffmpeg (handles ALL extensions).
    Called once per sample; array freed after return.
    """
    path = batch["path"]
    try:
        # librosa uses ffmpeg under the hood — works for mp4, m4a, ogg, mp3, wav …
        arr, _ = librosa.load(
            path,
            sr=SAMPLE_RATE,
            mono=True,
            dtype=np.float32
        )
    except Exception as e:
        # Return dummy silent frame if file is corrupt — trainer will see -100 label
        print(f"    Skipping corrupt file {os.path.basename(path)}: {e}")
        arr = np.zeros(SAMPLE_RATE, dtype=np.float32)   # 1 s silence
        batch["sentence"] = ""                          # empty → -100 mask

    # Mel spectrogram on CPU (80 × 3000)
    feats = processor.feature_extractor(
        arr, sampling_rate=SAMPLE_RATE, return_tensors="np"
    ).input_features[0]

    # Tokenise transcript
    ids = processor.tokenizer(
        batch["sentence"],
        padding=False,
        truncation=True,
        max_length=448,
    ).input_ids

    del arr   # explicit free before Arrow write
    return {"input_features": feats, "labels": ids}

def run_map(ds, split_name):
    ram = psutil.virtual_memory().used / 1e9
    print(f"\n  [{split_name}] {len(ds)} samples  RAM before: {ram:.1f} GB")
    out = ds.map(
        prepare_features,
        remove_columns=ds.column_names,   # drop 'path' and 'sentence' cols
        batched=False,                    # 1 sample at a time → O(1) RAM
        num_proc=NUM_MAP_WORKERS,
        writer_batch_size=WRITER_BATCH_SIZE,
        cache_file_name=os.path.join(ARROW_CACHE, f"{split_name}_features.arrow"),
        desc=f"{split_name} features",
    )
    gc.collect()
    ram = psutil.virtual_memory().used / 1e9
    mb  = (out.dataset_size or 0) / 1e6
    print(f"   Done.  RAM after: {ram:.1f} GB  |  Arrow on SSD: {mb:.0f} MB")
    return out

train_ds = run_map(train_ds, "train")
eval_ds  = run_map(eval_ds,  "eval")

gc.collect()
torch.cuda.empty_cache()
print("\n Step 5 complete — features on SSD, RAM freed")
print(f"   Final RAM: {psutil.virtual_memory().used/1e9:.1f} GB")

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...


  [train] 4572 samples  RAM before: 2.3 GB


train features (num_proc=2):   0%|          | 0/4572 [00:00<?, ? examples/s]

   Done.  RAM after: 2.4 GB  |  Arrow on SSD: 0 MB

  [eval] 509 samples  RAM before: 2.4 GB


eval features (num_proc=2):   0%|          | 0/509 [00:00<?, ? examples/s]

   Done.  RAM after: 2.4 GB  |  Arrow on SSD: 0 MB

 Step 5 complete — features on SSD, RAM freed
   Final RAM: 2.4 GB


In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Split inputs and labels
        input_features = ([{"input_features": f["input_features"]} for f in features])
        label_features = ([{"input_ids": f["labels"]} for f in features])

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Replace padding with -100 so loss ignores it
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        # Strip BOS token added by tokenizer
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

print(" Data collator ready")

 Data collator ready


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 7 — Load Whisper Medium + Inject LoRA manually
#
# Root-cause of the recurring TypeError:
#   get_peft_model() wraps the model in PeftModelForSeq2SeqLM whose forward()
#   HARDCODES input_ids=input_ids in the call to base_model — no amount of
#   trainer overrides can fix this because PEFT intercepts before our code runs.
#
# Solution: skip get_peft_model() entirely.
#   1. Load plain WhisperForConditionalGeneration
#   2. Use loralib to replace Q/V projections with LoRA linear layers
#   3. Mark only LoRA params as trainable — identical training behaviour,
#      zero PEFT wrapper issues.
# ─────────────────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
from transformers import WhisperForConditionalGeneration

# ── 1. Load base model ────────────────────────────────────────────────────────
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME, cache_dir=CACHE_DIR
)
model.config.forced_decoder_ids = None
model.config.suppress_tokens    = []
model.config.use_cache          = False   # required for gradient checkpointing

# ── 2. Simple LoRA linear (no external lib needed) ───────────────────────────
class LoRALinear(nn.Module):
    """Drop-in LoRA wrapper for nn.Linear. Adds A*B low-rank branch."""
    def __init__(self, linear: nn.Linear, r: int, alpha: int, dropout: float):
        super().__init__()
        self.linear   = linear          # frozen original weight
        self.r        = r
        self.scaling  = alpha / r
        self.lora_A   = nn.Linear(linear.in_features,  r, bias=False)
        self.lora_B   = nn.Linear(r, linear.out_features, bias=False)
        self.dropout  = nn.Dropout(dropout)
        # initialise: A ~ N(0,1), B = 0  → LoRA starts as identity delta
        nn.init.normal_(self.lora_A.weight)
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        return self.linear(x) + self.lora_B(self.lora_A(self.dropout(x))) * self.scaling

def inject_lora(model, r, alpha, dropout, target_suffixes=("q_proj", "v_proj")):
    """Replace matching Linear layers with LoRALinear in-place."""
    replaced = 0
    for name, module in list(model.named_modules()):
        for suffix in target_suffixes:
            if name.endswith(suffix) and isinstance(module, nn.Linear):
                # Navigate to parent and swap the attribute
                parts  = name.split(".")
                parent = model
                for p in parts[:-1]:
                    parent = getattr(parent, p)
                setattr(parent, parts[-1],
                        LoRALinear(module, r=r, alpha=alpha, dropout=dropout))
                replaced += 1
    return replaced

n = inject_lora(model, r=LORA_R, alpha=LORA_ALPHA, dropout=LORA_DROPOUT)
print(f"   Injected LoRA into {n} projection layers")

# ── 3. Freeze everything except LoRA params ───────────────────────────────────
total = trainable = 0
for name, param in model.named_parameters():
    is_lora = ("lora_A" in name or "lora_B" in name)
    param.requires_grad = is_lora
    total     += param.numel()
    trainable += param.numel() if is_lora else 0

print(f"   Trainable : {trainable:,}  /  Total : {total:,}  "
      f"({100*trainable/total:.2f}%)")

#  FIX: required when gradient_checkpointing=True with frozen base layers.
# Ensures the input embeddings always emit a grad-requiring tensor so
# the autograd graph is never broken during the checkpoint re-forward pass.
model.enable_input_require_grads()

model.generation_config.language = LANGUAGE.lower()
model.generation_config.task     = TASK
print(" Model ready (plain Whisper + manual LoRA — no PEFT wrapper)")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

   Injected LoRA into 72 projection layers
   Trainable : 14,155,776  /  Total : 255,890,688  (5.53%)
 Model ready (plain Whisper + manual LoRA — no PEFT wrapper)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 8 — Data collator
#  Instantiate DataCollatorSpeechSeq2SeqWithPadding
# ─────────────────────────────────────────────────────────────────────────────
import torch

# Instantiate the DataCollatorSpeechSeq2SeqWithPadding
# The 'processor' object is available from the feature extraction step (cell xdm_12pG0Fbd)
# The 'model' object is available from the previous step (cell h2auE_bp3o-8)

# Get the decoder_start_token_id from the model's generation config
decoder_start_token_id = model.config.decoder_start_token_id

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=decoder_start_token_id
)

print(" Data collator instantiated successfully!")

 Data collator instantiated successfully!


In [ ]:
import evaluate
import numpy as np

metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids   = pred.predictions
    label_ids  = pred.label_ids

    # Replace -100 → pad token
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": round(wer, 2)}

print(" WER metric ready")

 WER metric ready


In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # ── Steps ────────────────────────────────────────────────────────────────
    # ── Steps ────────────────────────────────────────────────────────────────
    # num_train_epochs removed — max_steps=2000 controls training length
    max_steps=MAX_STEPS,

    # ── Batch / Gradient ─────────────────────────────────────────────────────
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,        # saves ~4 GB VRAM

    # ── Optimiser ────────────────────────────────────────────────────────────
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    optim="adamw_torch_fused",          # faster and slightly less memory
    lr_scheduler_type="linear",

    # ── Precision ────────────────────────────────────────────────────────────
    fp16=FP16,

    # ── Evaluation / Saving ──────────────────────────────────────────────────
    evaluation_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,                 # keep only 2 checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    # ── Generation during eval ───────────────────────────────────────────────
    predict_with_generate=True,
    generation_max_length=225,

    # ── Misc ─────────────────────────────────────────────────────────────────
    logging_steps=25,
    report_to=["none"],                 # disable wandb (avoids prompts)
    push_to_hub=False,
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

print(" Training arguments set")

 Training arguments set


In [ ]:
import gc, torch
import shutil
import re
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, WhisperConfig, EarlyStoppingCallback, TrainerCallback, TrainerState # Import WhisperConfig, EarlyStoppingCallback, TrainerCallback, TrainerState
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import os
import numpy as np # For torch.serialization.add_safe_globals([np.dtypes.UInt32DType])

# Import necessary for model loading and LoRA injection
import torch.nn as nn
from transformers import WhisperForConditionalGeneration


# Re-define DataCollatorSpeechSeq2SeqWithPadding class to ensure it's in scope
# This definition is taken from previous cells.
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = ([{"input_features": f["input_features"]} for f in features])
        label_features = ([{"input_ids": f["labels"]} for f in features])

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

# --- Helper function to find the latest valid checkpoint and its step number ---
# This function was adapted from a previous scratchpad cell JwTi-3n8MLSj
def get_latest_checkpoint_and_step(output_dir):
    checkpoints = [] # Stores (actual_step, folder_path)
    if not os.path.exists(output_dir):
        print(f"  Warning: Output directory '{output_dir}' does not exist.")
        return None, 0

    checkpoint_dirs = [f for f in os.listdir(output_dir) if re.match(r"checkpoint-(\d+)", f) and os.path.isdir(os.path.join(output_dir, f))]
    if not checkpoint_dirs:
        print(f"  Warning: No checkpoint directories found in '{output_dir}'.")
        return None, 0

    for f_name in checkpoint_dirs:
        full_path = os.path.join(output_dir, f_name)
        trainer_state_path = os.path.join(full_path, "trainer_state.json")
        model_weights_path_bin = os.path.join(full_path, "pytorch_model.bin")
        model_weights_path_safetensors = os.path.join(full_path, "model.safetensors")

        # A checkpoint is considered valid if it has trainer_state.json AND model weights
        if os.path.exists(trainer_state_path) and \
           (os.path.exists(model_weights_path_bin) or os.path.exists(model_weights_path_safetensors)):
            try:
                state = TrainerState.load_from_json(trainer_state_path)
                actual_step = state.global_step
                checkpoints.append((actual_step, full_path))
                print(f"  Info: Found potential valid checkpoint '{f_name}' with global_step={actual_step}.")
            except Exception as e:
                print(f"  Warning: trainer_state.json in '{f_name}' is invalid or malformed. Skipping. Error: {e}")
                continue # Skip if trainer_state.json is malformed or can't be read
        else:
            print(f"  Warning: Checkpoint '{f_name}' is incomplete (missing trainer_state.json or model weights). Skipping.")

    if not checkpoints:
        print(f"  Error: No *valid and complete* checkpoints found in '{output_dir}'.")
        return None, 0

    checkpoints.sort(key=lambda x: x[0]) # Sort by actual_step from trainer_state.json
    latest_step, latest_path = checkpoints[-1]
    return latest_path, latest_step

# --- Parameters for additional training and early stopping ---
ADDITIONAL_STEPS = 2000 # Train for this many steps more
WER_TARGET = 39.99      # Aim for WER below 40%
EARLY_STOPPING_PATIENCE = 3 # Stop if no improvement for 3 evaluations (3 * EVAL_STEPS = 600 steps)
EARLY_STOPPING_THRESHOLD = 0.0 # Any improvement in WER is considered an improvement

# --- LoRA Model Re-initialization and Injection (Copied from h2auE_bp3o-8) ---
# ── Simple LoRA linear (no external lib needed) ───────────────────────────
class LoRALinear(nn.Module):
    """Drop-in LoRA wrapper for nn.Linear. Adds A*B low-rank branch."""
    def __init__(self, linear: nn.Linear, r: int, alpha: int, dropout: float):
        super().__init__()
        self.linear   = linear          # frozen original weight
        self.r        = r
        self.scaling  = alpha / r
        self.lora_A   = nn.Linear(linear.in_features,  r, bias=False)
        self.lora_B   = nn.Linear(r, linear.out_features, bias=False)
        self.dropout  = nn.Dropout(dropout)
        # initialise: A ~ N(0,1), B = 0  → LoRA starts as identity delta
        nn.init.normal_(self.lora_A.weight)
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        return self.linear(x) + self.lora_B(self.lora_A(self.dropout(x))) * self.scaling

def inject_lora(model, r, alpha, dropout, target_suffixes=("q_proj", "v_proj")):
    """Replace matching Linear layers with LoRALinear in-place."""
    replaced = 0
    for name, module in list(model.named_modules()):
        for suffix in target_suffixes:
            if name.endswith(suffix) and isinstance(module, nn.Linear):
                # Navigate to parent and swap the attribute
                parts  = name.split(".")
                parent = model
                for p in parts[:-1]:
                    parent = getattr(parent, p)
                setattr(parent, parts[-1],
                        LoRALinear(module, r=r, alpha=alpha, dropout=dropout))
                replaced += 1
    return replaced

# 1. Load the base Whisper model with current configuration
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME, cache_dir=CACHE_DIR
)
model.config.forced_decoder_ids = None
model.config.suppress_tokens    = []
model.config.use_cache          = False   # required for gradient checkpointing

# 2. Re-inject the LoRA layers with current hyperparameters
n = inject_lora(model, r=LORA_R, alpha=LORA_ALPHA, dropout=LORA_DROPOUT)
print(f"   Re-injected LoRA into {n} projection layers with current LORA_R={LORA_R}")

# 3. Freeze everything except LoRA params
# This part is crucial for LoRA training and was in cell h2auE_bp3o-8
total = trainable = 0
for name, param in model.named_parameters():
    is_lora = ("lora_A" in name or "lora_B" in name)
    param.requires_grad = is_lora
    total     += param.numel()
    trainable += param.numel() if is_lora else 0

print(f"   Trainable : {trainable:,}  /  Total : {total:,}  "\
      f"({100*trainable/total:.2f}%)")

# FIX: required when gradient_checkpointing=True with frozen base layers.
# Ensures the input embeddings always emit a grad-requiring tensor so
# the autograd graph is never broken during the checkpoint re-forward pass.
model.enable_input_require_grads()

model.generation_config.language = LANGUAGE.lower()
model.generation_config.task     = TASK
print(" Model ready (plain Whisper + manual LoRA — no PEFT wrapper)")
# --- End of LoRA Model Re-initialization and Injection ---


# --- Find the latest valid checkpoint and determine total steps ---
# User explicitly requested to resume from a specific checkpoint, but then asked to train from scratch.
# Therefore, we explicitly set resume_from to None to force starting from scratch.
print("    Training from scratch as requested, using current LoRA configuration.")
resume_from = None
NEW_TOTAL_MAX_STEPS = MAX_STEPS + ADDITIONAL_STEPS # MAX_STEPS from config cell + ADDITIONAL_STEPS
print(f"    Starting from scratch, targeting a total of {NEW_TOTAL_MAX_STEPS} steps.")

# The get_latest_checkpoint_and_step function is still defined but its result is ignored for `resume_from` in this specific fix.

print(f"\n{'='*60}")
print(f"    Continuing training for a total of {NEW_TOTAL_MAX_STEPS} steps.")
print(f"    Targeting a WER below {WER_TARGET:.2f}%. Early stopping patience: {EARLY_STOPPING_PATIENCE} evaluations ({EARLY_STOPPING_PATIENCE * EVAL_STEPS} steps).")
print(f"{'='*60}")

# --- FIX: UnpicklingError when resuming training ---
# This is required for torch.load to successfully load the random number generator state
# from the checkpoint when weights_only=True is the default (PyTorch 2.6+).
# Add specific numpy globals that might be present in the checkpoint.
torch.serialization.add_safe_globals([np.dtypes.UInt32DType, np._core.multiarray._reconstruct, np.ndarray, np.dtype])
# -----------------------------------------------------------------------------

# Assume MODEL_NAME, CACHE_DIR, processor, model, train_ds, eval_ds, compute_metrics,
# BATCH_SIZE, GRAD_ACCUM, LEARNING_RATE, WARMUP_STEPS, EVAL_STEPS, SAVE_STEPS, FP16
# are already defined in the global scope from previous cells.
# If running this cell independently, these would need to be defined/loaded first.

# Re-initialize DataCollator
whisper_cfg = WhisperConfig.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
decoder_start_id = whisper_cfg.decoder_start_token_id
del whisper_cfg

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=decoder_start_id
)

# Create new training arguments with the updated max_steps
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=NEW_TOTAL_MAX_STEPS, # Use the new total max steps
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    optim="adamw_torch_fused",
    lr_scheduler_type="linear",
    fp16=FP16,
    evaluation_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2, # Keep only 2 recent checkpoints as defined previously
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=225,
    logging_steps=25,
    report_to=["none"],
    push_to_hub=False,
    dataloader_num_workers=2,
    remove_unused_columns=False,
    # resume_from_checkpoint is passed directly to trainer.train()
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE, early_stopping_threshold=EARLY_STOPPING_THRESHOLD)] # Add early stopping
)

# Call trainer.train() with the explicitly specified checkpoint
trainer.train(resume_from_checkpoint=resume_from)

# Evaluate after the training to see the final WER
results = trainer.evaluate()
final_wer = results.get("eval_wer", 999.0)

print(f"\n{'='*60}")
if final_wer <= WER_TARGET:
    print(f"    Training completed! Final WER: {final_wer:.2f}% (achieved target below {WER_TARGET:.2f}%)")
elif trainer.state.stopped_early:
    print(f"    Training stopped early due to no improvement in WER for {EARLY_STOPPING_PATIENCE * EVAL_STEPS} steps. Final WER: {final_wer:.2f}%. Target below {WER_TARGET:.2f}% not yet met.")
    print("    Suggestions: Consider reviewing hyperparameters (learning rate, batch size, LoRA config) or increasing the dataset size.")
else:
    print(f"    Training completed up to {NEW_TOTAL_MAX_STEPS} steps. Final WER: {final_wer:.2f}%. Target below {WER_TARGET:.2f}% not met.")
print(f"  Checkpoints are saved in: {OUTPUT_DIR}")
print(f"{'='*60}")

# Free up memory
del trainer, training_args
gc.collect()
torch.cuda.empty_cache()


   Re-injected LoRA into 72 projection layers with current LORA_R=128
   Trainable : 14,155,776  /  Total : 255,890,688  (5.53%)
 Model ready (plain Whisper + manual LoRA — no PEFT wrapper)
    Training from scratch as requested, using current LoRA configuration.
    Starting from scratch, targeting a total of 12000 steps.

    Continuing training for a total of 12000 steps.
    Targeting a WER below 39.99%. Early stopping patience: 3 evaluations (600 steps).


max_steps is given, it will override any value given in num_train_epochs


Step,Training Loss,Validation Loss,Wer
200,1.838100,1.651758,90.270000
400,1.683800,1.523058,83.080000
600,1.575100,1.381691,76.770000
800,1.430300,1.308535,73.580000
1000,1.359000,1.280651,72.720000
1200,1.307900,1.230403,71.960000
1400,1.245900,1.167483,69.160000
1600,1.228000,1.156963,68.880000
1800,1.183200,1.123648,66.720000
2000,1.168400,1.105406,64.700000


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-cust

In [ ]:
import gc, torch
import shutil
import re
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, WhisperConfig, EarlyStoppingCallback, TrainerCallback, TrainerState # Import WhisperConfig, EarlyStoppingCallback, TrainerCallback, TrainerState
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import os
import numpy as np # For torch.serialization.add_safe_globals([np.dtypes.UInt32DType])

# Import necessary for model loading and LoRA injection
import torch.nn as nn
from transformers import WhisperForConditionalGeneration

# Re-define config variables (copied from cell 0SEVKD90zsn1/b9ef2006 for robustness)
DRIVE_ROOT = "/content/drive/MyDrive/Audios_Phusto"
OUTPUT_DIR = "/content/drive/MyDrive/Audios_Phusto/whisper-pashto-lora"
CACHE_DIR  = "/content/cache"
MODEL_NAME = "openai/whisper-small"
LANGUAGE   = "Pashto"
TASK       = "transcribe"
BATCH_SIZE = 4
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4 # Reverting to original LR from 0SEVKD90zsn1
WARMUP_STEPS = 200
EVAL_STEPS = 200
SAVE_STEPS = 200
FP16       = True
# Note: MAX_STEPS in config cell is 3000, but we will define NEW_TOTAL_MAX_STEPS for this specific training run.

# LoRA hyperparameters (from 0SEVKD90zsn1)
LORA_R           = 128
LORA_ALPHA       = 256
LORA_DROPOUT     = 0.15

# Re-define DataCollatorSpeechSeq2SeqWithPadding class to ensure it's in scope
# This definition is taken from previous cells.
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = ([{"input_features": f["input_features"]} for f in features])
        label_features = ([{"input_ids": f["labels"]} for f in features])

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

# --- LoRA Model Re-initialization and Injection (Copied from h2auE_bp3o-8) ---
# ── Simple LoRA linear (no external lib needed) ───────────────────────────
class LoRALinear(nn.Module):
    """Drop-in LoRA wrapper for nn.Linear. Adds A*B low-rank branch."""
    def __init__(self, linear: nn.Linear, r: int, alpha: int, dropout: float):
        super().__init__()
        self.linear   = linear          # frozen original weight
        self.r        = r
        self.scaling  = alpha / r
        self.lora_A   = nn.Linear(linear.in_features,  r, bias=False)
        self.lora_B   = nn.Linear(r, linear.out_features, bias=False)
        self.dropout  = nn.Dropout(dropout)
        # initialise: A ~ N(0,1), B = 0  → LoRA starts as identity delta
        nn.init.normal_(self.lora_A.weight)
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        return self.linear(x) + self.lora_B(self.lora_A(self.dropout(x))) * self.scaling

def inject_lora(model, r, alpha, dropout, target_suffixes=("q_proj", "v_proj")):
    """Replace matching Linear layers with LoRALinear in-place."""
    replaced = 0
    for name, module in list(model.named_modules()):
        for suffix in target_suffixes:
            if name.endswith(suffix) and isinstance(module, nn.Linear):
                # Navigate to parent and swap the attribute
                parts  = name.split(".")
                parent = model
                for p in parts[:-1]:
                    parent = getattr(parent, p)
                setattr(parent, parts[-1],
                        LoRALinear(module, r=r, alpha=alpha, dropout=dropout))
                replaced += 1
    return replaced

# 1. Load the base Whisper model with current configuration
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME, cache_dir=CACHE_DIR
)
model.config.forced_decoder_ids = None
model.config.suppress_tokens    = []
model.config.use_cache          = False   # required for gradient checkpointing

# 2. Re-inject the LoRA layers with current hyperparameters
n = inject_lora(model, r=LORA_R, alpha=LORA_ALPHA, dropout=LORA_DROPOUT)
print(f"   Re-injected LoRA into {n} projection layers with current LORA_R={LORA_R}")

# 3. Freeze everything except LoRA params
total = trainable = 0
for name, param in model.named_parameters():
    is_lora = ("lora_A" in name or "lora_B" in name)
    param.requires_grad = is_lora
    total     += param.numel()
    trainable += param.numel() if is_lora else 0

print(f"   Trainable : {trainable:,}  /  Total : {total:,}  "\
      f"({100*trainable/total:.2f}%)")

# FIX: required when gradient_checkpointing=True with frozen base layers.
# Ensures the input embeddings always emit a grad-requiring tensor so
# the autograd graph is never broken during the checkpoint re-forward pass.
model.enable_input_require_grads()

model.generation_config.language = LANGUAGE.lower()
model.generation_config.task     = TASK
print(" Model ready (plain Whisper + manual LoRA — no PEFT wrapper)")
# --- End of LoRA Model Re-initialization and Injection ---

# --- Parameters for additional training and early stopping ---
RESUME_FROM_STEP = 2000 # Explicitly requested to resume from this step
NEW_TOTAL_MAX_STEPS = 4000 # Explicitly requested to train until this step
WER_TARGET = 39.99      # Aim for WER below 40%
EARLY_STOPPING_PATIENCE = 3 # Stop if no improvement for 3 evaluations (3 * EVAL_STEPS = 600 steps)
EARLY_STOPPING_THRESHOLD = 0.0 # Any improvement in WER is considered an improvement

# Construct the path to the checkpoint to resume from
resume_from_checkpoint_path = os.path.join(OUTPUT_DIR, f"checkpoint-{RESUME_FROM_STEP}")

if not os.path.exists(resume_from_checkpoint_path):
    print(f"Error: Checkpoint path '{resume_from_checkpoint_path}' does not exist. Cannot resume from specified checkpoint.")
    print("Please ensure the checkpoint exists or adjust RESUME_FROM_STEP.")
    # Optionally, you could exit here or fall back to training from scratch
    # For this request, we will assume it should exist based on user's prompt.
    raise FileNotFoundError(f"Checkpoint {resume_from_checkpoint_path} not found.")

print(f"\n{'='*60}")
print(f"    Resuming training from step {RESUME_FROM_STEP} and targeting total {NEW_TOTAL_MAX_STEPS} steps.")
print(f"    Targeting a WER below {WER_TARGET:.2f}%. Early stopping patience: {EARLY_STOPPING_PATIENCE} evaluations ({EARLY_STOPPING_PATIENCE * EVAL_STEPS} steps).")
print(f"{'='*60}")

# --- FIX: UnpicklingError when resuming training ---
# This is required for torch.load to successfully load the random number generator state
# from the checkpoint when weights_only=True is the default (PyTorch 2.6+).
# Add specific numpy globals that might be present in the checkpoint.
torch.serialization.add_safe_globals([np.dtypes.UInt32DType, np._core.multiarray._reconstruct, np.ndarray, np.dtype])
# -----------------------------------------------------------------------------

# Assume processor, train_ds, eval_ds, compute_metrics are available from previous cells

# Re-initialize DataCollator
whisper_cfg = WhisperConfig.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
decoder_start_id = whisper_cfg.decoder_start_token_id
del whisper_cfg

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=decoder_start_id
)

# Create new training arguments with the updated max_steps
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=NEW_TOTAL_MAX_STEPS, # Use the new total max steps
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    optim="adamw_torch_fused",
    lr_scheduler_type="linear",
    fp16=FP16,
    evaluation_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2, # Keep only 2 recent checkpoints as defined previously
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=225,
    logging_steps=25,
    report_to=["none"],
    push_to_hub=False,
    dataloader_num_workers=2,
    remove_unused_columns=False,
    # resume_from_checkpoint is passed directly to trainer.train()
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE, early_stopping_threshold=EARLY_STOPPING_THRESHOLD)] # Add early stopping
)

# Call trainer.train() with the explicitly specified checkpoint
trainer.train(resume_from_checkpoint=resume_from_checkpoint_path)

# Evaluate after the training to see the final WER
results = trainer.evaluate()
final_wer = results.get("eval_wer", 999.0)

print(f"\n{'='*60}")
if final_wer <= WER_TARGET:
    print(f"    Training completed! Final WER: {final_wer:.2f}% (achieved target below {WER_TARGET:.2f}%)")
elif trainer.state.stopped_early:
    print(f"    Training stopped early due to no improvement in WER for {EARLY_STOPPING_PATIENCE * EVAL_STEPS} steps. Final WER: {final_wer:.2f}%. Target below {WER_TARGET:.2f}% not yet met.")
    print("    Suggestions: Consider reviewing hyperparameters (learning rate, batch size, LoRA config) or increasing the dataset size.")
else:
    print(f"    Training completed up to {NEW_TOTAL_MAX_STEPS} steps. Final WER: {final_wer:.2f}%. Target below {WER_TARGET:.2f}% not met.")
print(f"  Checkpoints are saved in: {OUTPUT_DIR}")
print(f"{'='*60}")

# Free up memory
del trainer, training_args
gc.collect()
torch.cuda.empty_cache()


   Re-injected LoRA into 72 projection layers with current LORA_R=128
   Trainable : 14,155,776  /  Total : 255,890,688  (5.53%)
 Model ready (plain Whisper + manual LoRA — no PEFT wrapper)

    Resuming training from step 2000 and targeting total 4000 steps.
    Targeting a WER below 39.99%. Early stopping patience: 3 evaluations (600 steps).


max_steps is given, it will override any value given in num_train_epochs
There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Step,Training Loss,Validation Loss,Wer
2200,1.038500,0.956224,60.660000
2400,0.993700,0.945852,59.930000
2600,0.902500,0.924690,59.430000
2800,0.947000,0.903098,58.460000
3000,0.874300,0.897413,57.950000
3200,0.840400,0.882302,57.710000
3400,0.817900,0.875356,57.530000
3600,0.787600,0.868356,56.830000
3800,0.769700,0.862031,56.420000
4000,0.738800,0.857365,56.170000


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-cust

AttributeError: 'TrainerState' object has no attribute 'stopped_early'

In [ ]:
import gc, torch
import shutil
import re
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, WhisperConfig, EarlyStoppingCallback, TrainerCallback, TrainerState # Import WhisperConfig, EarlyStoppingCallback, TrainerCallback, TrainerState
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import os
import numpy as np # For torch.serialization.add_safe_globals([np.dtypes.UInt32DType])

# Import necessary for model loading and LoRA injection
import torch.nn as nn
from transformers import WhisperForConditionalGeneration

# Re-define config variables (copied from cell 0SEVKD90zsn1 for robustness)
# Ensure these match your initial setup or are correctly updated.
DRIVE_ROOT = "/content/drive/MyDrive/Audios_Phusto"
OUTPUT_DIR = "/content/drive/MyDrive/Audios_Phusto/whisper-pashto-lora"
CACHE_DIR  = "/content/cache"
MODEL_NAME = "openai/whisper-small"
LANGUAGE   = "Pashto"
TASK       = "transcribe"
BATCH_SIZE = 4
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4 # Reverting to original LR from 0SEVKD90zsn1
WARMUP_STEPS = 200
EVAL_STEPS = 200
SAVE_STEPS = 200
FP16       = True

# LoRA hyperparameters (from 0SEVKD90zsn1)
LORA_R           = 128
LORA_ALPHA       = 256
LORA_DROPOUT     = 0.15

# Re-define DataCollatorSpeechSeq2SeqWithPadding class to ensure it's in scope
# This definition is taken from previous cells.
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = ([{"input_features": f["input_features"]} for f in features])
        label_features = ([{"input_ids": f["labels"]} for f in features])

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

# --- LoRA Model Re-initialization and Injection (Copied from h2auE_bp3o-8) ---
# This ensures the model is correctly initialized and LoRA layers are in place.
class LoRALinear(nn.Module):
    """Drop-in LoRA wrapper for nn.Linear. Adds A*B low-rank branch."""
    def __init__(self, linear: nn.Linear, r: int, alpha: int, dropout: float):
        super().__init__()
        self.linear   = linear          # frozen original weight
        self.r        = r
        self.scaling  = alpha / r
        self.lora_A   = nn.Linear(linear.in_features,  r, bias=False)
        self.lora_B   = nn.Linear(r, linear.out_features, bias=False)
        self.dropout  = nn.Dropout(dropout)
        # initialise: A ~ N(0,1), B = 0  → LoRA starts as identity delta
        nn.init.normal_(self.lora_A.weight)
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        return self.linear(x) + self.lora_B(self.lora_A(self.dropout(x))) * self.scaling

def inject_lora(model, r, alpha, dropout, target_suffixes=("q_proj", "v_proj")):
    """Replace matching Linear layers with LoRALinear in-place."""
    replaced = 0
    for name, module in list(model.named_modules()):
        for suffix in target_suffixes:
            if name.endswith(suffix) and isinstance(module, nn.Linear):
                # Navigate to parent and swap the attribute
                parts  = name.split(".")
                parent = model
                for p in parts[:-1]:
                    parent = getattr(parent, p)
                setattr(parent, parts[-1],
                        LoRALinear(module, r=r, alpha=alpha, dropout=dropout))
                replaced += 1
    return replaced

# 1. Load the base Whisper model with current configuration
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME, cache_dir=CACHE_DIR
)
model.config.forced_decoder_ids = None
model.config.suppress_tokens    = []
model.config.use_cache          = False   # required for gradient checkpointing

# 2. Re-inject the LoRA layers with current hyperparameters
n = inject_lora(model, r=LORA_R, alpha=LORA_ALPHA, dropout=LORA_DROPOUT)
print(f"   Re-injected LoRA into {n} projection layers with current LORA_R={LORA_R}")

# 3. Freeze everything except LoRA params
total = trainable = 0
for name, param in model.named_parameters():
    is_lora = ("lora_A" in name or "lora_B" in name)
    param.requires_grad = is_lora
    total     += param.numel()
    trainable += param.numel() if is_lora else 0

print(f"   Trainable : {trainable:,}  /  Total : {total:,}  "\
      f"({100*trainable/total:.2f}%)")

# FIX: required when gradient_checkpointing=True with frozen base layers.
# Ensures the input embeddings always emit a grad-requiring tensor so
# the autograd graph is never broken during the checkpoint re-forward pass.
model.enable_input_require_grads()

model.generation_config.language = LANGUAGE.lower()
model.generation_config.task     = TASK
print(" Model ready (plain Whisper + manual LoRA — no PEFT wrapper)")
# --- End of LoRA Model Re-initialization and Injection ---

# --- Parameters for additional training and early stopping ---
RESUME_FROM_STEP = 4000 # Resume from this step as requested
ADDITIONAL_STEPS = 2000 # Additional steps for training as requested
# Calculate the new total max steps based on the resume step and additional steps.
# If previous training ran up to 4000 steps, and we add 2000, new total is 6000.
NEW_TOTAL_MAX_STEPS = RESUME_FROM_STEP + ADDITIONAL_STEPS
WER_TARGET = 39.99      # Aim for WER below 40%
EARLY_STOPPING_PATIENCE = 3 # Stop if no improvement for 3 evaluations (3 * EVAL_STEPS = 600 steps)
EARLY_STOPPING_THRESHOLD = 0.0 # Any improvement in WER is considered an improvement (allows continuing even if target met but still improving)

# Construct the path to the checkpoint to resume from
resume_from_checkpoint_path = os.path.join(OUTPUT_DIR, f"checkpoint-{RESUME_FROM_STEP}")

if not os.path.exists(resume_from_checkpoint_path):
    print(f"Error: Checkpoint path '{resume_from_checkpoint_path}' does not exist. Cannot resume from specified checkpoint.")
    print("Please ensure the checkpoint exists or adjust RESUME_FROM_STEP.")
    raise FileNotFoundError(f"Checkpoint {resume_from_checkpoint_path} not found.")

print(f"\n{'='*60}")
print(f"    Resuming training from step {RESUME_FROM_STEP} and targeting total {NEW_TOTAL_MAX_STEPS} steps.")
print(f"    Targeting a WER below {WER_TARGET:.2f}%. Early stopping patience: {EARLY_STOPPING_PATIENCE} evaluations ({EARLY_STOPPING_PATIENCE * EVAL_STEPS} steps).")
print(f"{'='*60}")

# --- FIX: UnpicklingError when resuming training ---
# This is required for torch.load to successfully load the random number generator state
# from the checkpoint when weights_only=True is the default (PyTorch 2.6+).
# Add specific numpy globals that might be present in the checkpoint.
torch.serialization.add_safe_globals([np.dtypes.UInt32DType, np._core.multiarray._reconstruct, np.ndarray, np.dtype])
# -----------------------------------------------------------------------------

# Assume processor, train_ds, eval_ds, compute_metrics are available from previous cells

# Re-initialize DataCollator
whisper_cfg = WhisperConfig.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
decoder_start_id = whisper_cfg.decoder_start_token_id
del whisper_cfg

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=decoder_start_id
)

# Create new training arguments with the updated max_steps
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=NEW_TOTAL_MAX_STEPS, # Use the new total max steps
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    optim="adamw_torch_fused",
    lr_scheduler_type="linear",
    fp16=FP16,
    evaluation_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2, # Keep only 2 recent checkpoints as defined previously
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=225,
    logging_steps=25,
    report_to=["none"],
    push_to_hub=False,
    dataloader_num_workers=2,
    remove_unused_columns=False,
    # resume_from_checkpoint is passed directly to trainer.train()
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE, early_stopping_threshold=EARLY_STOPPING_THRESHOLD)] # Add early stopping
)

# Call trainer.train() with the explicitly specified checkpoint
trainer.train(resume_from_checkpoint=resume_from_checkpoint_path)

# Evaluate after the training to see the final WER
results = trainer.evaluate()
final_wer = results.get("eval_wer", 999.0)

print(f"\n{'='*60}")
if final_wer <= WER_TARGET:
    print(f"    Training completed! Final WER: {final_wer:.2f}% (achieved target below {WER_TARGET:.2f}%")
elif trainer.state.stopped_early:
    print(f"    Training stopped early due to no improvement in WER for {EARLY_STOPPING_PATIENCE * EVAL_STEPS} steps. Final WER: {final_wer:.2f}%. Target below {WER_TARGET:.2f}% not yet met.")
    print("    Suggestions: Consider reviewing hyperparameters (learning rate, batch size, LoRA config) or increasing the dataset size.")
else:
    print(f"    Training completed up to {NEW_TOTAL_MAX_STEPS} steps. Final WER: {final_wer:.2f}%. Target below {WER_TARGET:.2f}% not met.")
print(f"  Checkpoints are saved in: {OUTPUT_DIR}")
print(f"{'='*60}")

# Free up memory
del trainer, training_args
gc.collect()
torch.cuda.empty_cache()

   Re-injected LoRA into 72 projection layers with current LORA_R=128
   Trainable : 14,155,776  /  Total : 255,890,688  (5.53%)
 Model ready (plain Whisper + manual LoRA — no PEFT wrapper)

    Resuming training from step 4000 and targeting total 6000 steps.
    Targeting a WER below 39.99%. Early stopping patience: 3 evaluations (600 steps).


max_steps is given, it will override any value given in num_train_epochs
There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Step,Training Loss,Validation Loss,Wer
4200,0.827400,0.793911,55.860000
4400,0.782600,0.793685,55.410000
4600,0.737600,0.788710,56.110000
4800,0.785800,0.786001,55.220000
5000,0.754400,0.777884,54.920000
5200,0.733600,0.776408,54.360000
5400,0.736500,0.768631,53.880000
5600,0.717400,0.765603,53.610000
5800,0.682000,0.765472,53.700000
6000,0.667200,0.762920,53.670000


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-cust

AttributeError: 'TrainerState' object has no attribute 'stopped_early'

In [ ]:
import gc, torch
import shutil
import re
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, WhisperConfig, EarlyStoppingCallback, TrainerCallback, TrainerState # Import WhisperConfig, EarlyStoppingCallback, TrainerCallback, TrainerState
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import os
import numpy as np # For torch.serialization.add_safe_globals([np.dtypes.UInt32DType])

# Import necessary for model loading and LoRA injection
import torch.nn as nn
from transformers import WhisperForConditionalGeneration

# Re-define config variables (copied from cell 0SEVKD90zsn1 for robustness)
# Ensure these match your initial setup or are correctly updated.
DRIVE_ROOT = "/content/drive/MyDrive/Audios_Phusto"
OUTPUT_DIR = "/content/drive/MyDrive/Audios_Phusto/whisper-pashto-lora"
CACHE_DIR  = "/content/cache"
MODEL_NAME = "openai/whisper-small"
LANGUAGE   = "Pashto"
TASK       = "transcribe"
BATCH_SIZE = 4
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4 # Reverting to original LR from 0SEVKD90zsn1
WARMUP_STEPS = 200
EVAL_STEPS = 200
SAVE_STEPS = 200
FP16       = True

# LoRA hyperparameters (from 0SEVKD90zsn1)
LORA_R           = 128
LORA_ALPHA       = 256
LORA_DROPOUT     = 0.15

# Re-define DataCollatorSpeechSeq2SeqWithPadding class to ensure it's in scope
# This definition is taken from previous cells.
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = ([{"input_features": f["input_features"]} for f in features])
        label_features = ([{"input_ids": f["labels"]} for f in features])

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

# --- LoRA Model Re-initialization and Injection (Copied from h2auE_bp3o-8) ---
# This ensures the model is correctly initialized and LoRA layers are in place.
class LoRALinear(nn.Module):
    """Drop-in LoRA wrapper for nn.Linear. Adds A*B low-rank branch."""
    def __init__(self, linear: nn.Linear, r: int, alpha: int, dropout: float):
        super().__init__()
        self.linear   = linear          # frozen original weight
        self.r        = r
        self.scaling  = alpha / r
        self.lora_A   = nn.Linear(linear.in_features,  r, bias=False)
        self.lora_B   = nn.Linear(r, linear.out_features, bias=False)
        self.dropout  = nn.Dropout(dropout)
        # initialise: A ~ N(0,1), B = 0  → LoRA starts as identity delta
        nn.init.normal_(self.lora_A.weight)
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        return self.linear(x) + self.lora_B(self.lora_A(self.dropout(x))) * self.scaling

def inject_lora(model, r, alpha, dropout, target_suffixes=("q_proj", "v_proj")):
    """Replace matching Linear layers with LoRALinear in-place."""
    replaced = 0
    for name, module in list(model.named_modules()):
        for suffix in target_suffixes:
            if name.endswith(suffix) and isinstance(module, nn.Linear):
                # Navigate to parent and swap the attribute
                parts  = name.split(".")
                parent = model
                for p in parts[:-1]:
                    parent = getattr(parent, p)
                setattr(parent, parts[-1],
                        LoRALinear(module, r=r, alpha=alpha, dropout=dropout))
                replaced += 1
    return replaced

# 1. Load the base Whisper model with current configuration
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME, cache_dir=CACHE_DIR
)
model.config.forced_decoder_ids = None
model.config.suppress_tokens    = []
model.config.use_cache          = False   # required for gradient checkpointing

# 2. Re-inject the LoRA layers with current hyperparameters
n = inject_lora(model, r=LORA_R, alpha=LORA_ALPHA, dropout=LORA_DROPOUT)
print(f"   Re-injected LoRA into {n} projection layers with current LORA_R={LORA_R}")

# 3. Freeze everything except LoRA params
total = trainable = 0
for name, param in model.named_parameters():
    is_lora = ("lora_A" in name or "lora_B" in name)
    param.requires_grad = is_lora
    total     += param.numel()
    trainable += param.numel() if is_lora else 0

print(f"   Trainable : {trainable:,}  /  Total : {total:,}  "\
      f"({100*trainable/total:.2f}%)")

# FIX: required when gradient_checkpointing=True with frozen base layers.
# Ensures the input embeddings always emit a grad-requiring tensor so
# the autograd graph is never broken during the checkpoint re-forward pass.
model.enable_input_require_grads()

model.generation_config.language = LANGUAGE.lower()
model.generation_config.task     = TASK
print(" Model ready (plain Whisper + manual LoRA — no PEFT wrapper)")
# --- End of LoRA Model Re-initialization and Injection ---

# --- Parameters for additional training and early stopping ---
RESUME_FROM_STEP = 6000 # Resume from this step as requested
ADDITIONAL_STEPS = 1000 # Additional steps for training as requested
# Calculate the new total max steps based on the resume step and additional steps.
# If previous training ran up to 4000 steps, and we add 2000, new total is 6000.
NEW_TOTAL_MAX_STEPS = RESUME_FROM_STEP + ADDITIONAL_STEPS
WER_TARGET = 39.99      # Aim for WER below 40%
EARLY_STOPPING_PATIENCE = 3 # Stop if no improvement for 3 evaluations (3 * EVAL_STEPS = 600 steps)
EARLY_STOPPING_THRESHOLD = 0.0 # Any improvement in WER is considered an improvement (allows continuing even if target met but still improving)

# Construct the path to the checkpoint to resume from
resume_from_checkpoint_path = os.path.join(OUTPUT_DIR, f"checkpoint-{RESUME_FROM_STEP}")

if not os.path.exists(resume_from_checkpoint_path):
    print(f"Error: Checkpoint path '{resume_from_checkpoint_path}' does not exist. Cannot resume from specified checkpoint.")
    print("Please ensure the checkpoint exists or adjust RESUME_FROM_STEP.")
    raise FileNotFoundError(f"Checkpoint {resume_from_checkpoint_path} not found.")

print(f"\n{'='*60}")
print(f"    Resuming training from step {RESUME_FROM_STEP} and targeting total {NEW_TOTAL_MAX_STEPS} steps.")
print(f"    Targeting a WER below {WER_TARGET:.2f}%. Early stopping patience: {EARLY_STOPPING_PATIENCE} evaluations ({EARLY_STOPPING_PATIENCE * EVAL_STEPS} steps).")
print(f"{'='*60}")

# --- FIX: UnpicklingError when resuming training ---
# This is required for torch.load to successfully load the random number generator state
# from the checkpoint when weights_only=True is the default (PyTorch 2.6+).
# Add specific numpy globals that might be present in the checkpoint.
torch.serialization.add_safe_globals([np.dtypes.UInt32DType, np._core.multiarray._reconstruct, np.ndarray, np.dtype])
# -----------------------------------------------------------------------------

# Assume processor, train_ds, eval_ds, compute_metrics are available from previous cells

# Re-initialize DataCollator
whisper_cfg = WhisperConfig.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
decoder_start_id = whisper_cfg.decoder_start_token_id
del whisper_cfg

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=decoder_start_id
)

# Create new training arguments with the updated max_steps
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=NEW_TOTAL_MAX_STEPS, # Use the new total max steps
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    optim="adamw_torch_fused",
    lr_scheduler_type="linear",
    fp16=FP16,
    evaluation_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2, # Keep only 2 recent checkpoints as defined previously
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=225,
    logging_steps=25,
    report_to=["none"],
    push_to_hub=False,
    dataloader_num_workers=2,
    remove_unused_columns=False,
    # resume_from_checkpoint is passed directly to trainer.train()
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE, early_stopping_threshold=EARLY_STOPPING_THRESHOLD)] # Add early stopping
)

# Call trainer.train() with the explicitly specified checkpoint
trainer.train(resume_from_checkpoint=resume_from_checkpoint_path)

# Evaluate after the training to see the final WER
results = trainer.evaluate()
final_wer = results.get("eval_wer", 999.0)

print(f"\n{'='*60}")
if final_wer <= WER_TARGET:
    print(f"    Training completed! Final WER: {final_wer:.2f}% (achieved target below {WER_TARGET:.2f}%)")
elif trainer.state.global_step < NEW_TOTAL_MAX_STEPS: # Check if training stopped before max_steps
    print(f"    Training stopped early due to no improvement in WER for {EARLY_STOPPING_PATIENCE * EVAL_STEPS} steps. Final WER: {final_wer:.2f}%. Target below {WER_TARGET:.2f}% not yet met.")
    print("    Suggestions: Consider reviewing hyperparameters (learning rate, batch size, LoRA config) or increasing the dataset size.")
else:
    print(f"    Training completed up to {NEW_TOTAL_MAX_STEPS} steps. Final WER: {final_wer:.2f}%. Target below {WER_TARGET:.2f}% not met.")
print(f"  Checkpoints are saved in: {OUTPUT_DIR}")
print(f"{'='*60}")

# Free up memory
del trainer, training_args
gc.collect()
torch.cuda.empty_cache()

   Re-injected LoRA into 72 projection layers with current LORA_R=128
   Trainable : 14,155,776  /  Total : 255,890,688  (5.53%)
 Model ready (plain Whisper + manual LoRA — no PEFT wrapper)

    Resuming training from step 6000 and targeting total 7000 steps.
    Targeting a WER below 39.99%. Early stopping patience: 3 evaluations (600 steps).


max_steps is given, it will override any value given in num_train_epochs
There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Step,Training Loss,Validation Loss,Wer
6200,0.737000,0.708265,51.180000
6400,0.661600,0.712025,51.230000
6600,0.633100,0.710670,50.690000
6800,0.688000,0.708430,50.580000
7000,0.660300,0.707116,50.460000


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-cust


    Training completed up to 7000 steps. Final WER: 50.46%. Target below 39.99% not met.
  Checkpoints are saved in: /content/drive/MyDrive/Audios_Phusto/whisper-pashto-lora


In [ ]:
import gc, torch
import shutil
import re
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, WhisperConfig, EarlyStoppingCallback, TrainerCallback, TrainerState # Import WhisperConfig, EarlyStoppingCallback, TrainerCallback, TrainerState
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import os
import numpy as np # For torch.serialization.add_safe_globals([np.dtypes.UInt32DType])

# Import necessary for model loading and LoRA injection
import torch.nn as nn
from transformers import WhisperForConditionalGeneration, WhisperProcessor

# Re-define config variables (copied from cell 0SEVKD90zsn1 for robustness)
# Ensure these match your initial setup or are correctly updated.
DRIVE_ROOT = "/content/drive/MyDrive/Audios_Phusto"
OUTPUT_DIR = "/content/drive/MyDrive/Audios_Phusto/whisper-pashto-lora"
CACHE_DIR  = "/content/cache"
MODEL_NAME = "openai/whisper-small"
LANGUAGE   = "Pashto"
TASK       = "transcribe"
BATCH_SIZE = 4
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4 # Reverting to original LR from 0SEVKD90zsn1
WARMUP_STEPS = 200
EVAL_STEPS = 200
SAVE_STEPS = 200
FP16       = True

# LoRA hyperparameters (from 0SEVKD90zsn1)
LORA_R           = 128
LORA_ALPHA       = 256
LORA_DROPOUT     = 0.15

# Re-define DataCollatorSpeechSeq2SeqWithPadding class to ensure it's in scope
# This definition is taken from previous cells.
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = ([{"input_features": f["input_features"]} for f in features])
        label_features = ([{"input_ids": f["labels"]} for f in features])

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

# --- LoRA Model Re-initialization and Injection (Copied from h2auE_bp3o-8) ---
# This ensures the model is correctly initialized and LoRA layers are in place.
class LoRALinear(nn.Module):
    """Drop-in LoRA wrapper for nn.Linear. Adds A*B low-rank branch."""
    def __init__(self, linear: nn.Linear, r: int, alpha: int, dropout: float):
        super().__init__()
        self.linear   = linear          # frozen original weight
        self.r        = r
        self.scaling  = alpha / r
        self.lora_A   = nn.Linear(linear.in_features,  r, bias=False)
        self.lora_B   = nn.Linear(r, linear.out_features, bias=False)
        self.dropout  = nn.Dropout(dropout)
        # initialise: A ~ N(0,1), B = 0  → LoRA starts as identity delta
        nn.init.normal_(self.lora_A.weight)
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        return self.linear(x) + self.lora_B(self.lora_A(self.dropout(x))) * self.scaling

def inject_lora(model, r, alpha, dropout, target_suffixes=("q_proj", "v_proj")):
    """Replace matching Linear layers with LoRALinear in-place."""
    replaced = 0
    for name, module in list(model.named_modules()):
        for suffix in target_suffixes:
            if name.endswith(suffix) and isinstance(module, nn.Linear):
                # Navigate to parent and swap the attribute
                parts  = name.split(".")
                parent = model
                for p in parts[:-1]:
                    parent = getattr(parent, p)
                setattr(parent, parts[-1],
                        LoRALinear(module, r=r, alpha=alpha, dropout=dropout))
                replaced += 1
    return replaced

# Re-initialize processor if it's not defined (copied from In75AymFu6QS)
if 'processor' not in locals() and 'processor' not in globals():
    print("WARNING: 'processor' not found, re-initializing from pre-trained model.")
    processor = WhisperProcessor.from_pretrained(
        MODEL_NAME, language=LANGUAGE, task=TASK, cache_dir=CACHE_DIR
    )

# 1. Load the base Whisper model with current configuration
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME, cache_dir=CACHE_DIR
)
model.config.forced_decoder_ids = None
model.config.suppress_tokens    = []
model.config.use_cache          = False   # required for gradient checkpointing

# 2. Re-inject the LoRA layers with current hyperparameters
n = inject_lora(model, r=LORA_R, alpha=LORA_ALPHA, dropout=LORA_DROPOUT)
print(f"   Re-injected LoRA into {n} projection layers with current LORA_R={LORA_R}")

# 3. Freeze everything except LoRA params
total = trainable = 0
for name, param in model.named_parameters():
    is_lora = ("lora_A" in name or "lora_B" in name)
    param.requires_grad = is_lora
    total     += param.numel()
    trainable += param.numel() if is_lora else 0

print(f"   Trainable : {trainable:,}  /  Total : {total:,}  "\
      f"({100*trainable/total:.2f}%)")

# FIX: required when gradient_checkpointing=True with frozen base layers.
# Ensures the input embeddings always emit a grad-requiring tensor so
# the autograd graph is never broken during the checkpoint re-forward pass.
model.enable_input_require_grads()

model.generation_config.language = LANGUAGE.lower()
model.generation_config.task     = TASK
print(" Model ready (plain Whisper + manual LoRA — no PEFT wrapper)")
# --- End of LoRA Model Re-initialization and Injection ---

# --- Parameters for additional training and early stopping ---
RESUME_FROM_STEP = 7000 # Resume from this step as requested
ADDITIONAL_STEPS = 1000 # Additional steps for training as requested (changed from 3000)
# Calculate the new total max steps based on the resume step and additional steps.
NEW_TOTAL_MAX_STEPS = RESUME_FROM_STEP + ADDITIONAL_STEPS
WER_TARGET = 39.99      # Aim for WER below 40%
EARLY_STOPPING_PATIENCE = 3 # Stop if no improvement for 3 evaluations (3 * EVAL_STEPS = 600 steps)
EARLY_STOPPING_THRESHOLD = 0.0 # Any improvement in WER is considered an improvement

# Construct the path to the checkpoint to resume from
resume_from_checkpoint_path = os.path.join(OUTPUT_DIR, f"checkpoint-{RESUME_FROM_STEP}")

if not os.path.exists(resume_from_checkpoint_path):
    print(f"Error: Checkpoint path '{resume_from_checkpoint_path}' does not exist. Cannot resume from specified checkpoint.")
    print("Please ensure the checkpoint exists or adjust RESUME_FROM_STEP.")
    raise FileNotFoundError(f"Checkpoint {resume_from_checkpoint_path} not found.")

print(f"\n{'='*60}")
print(f"    Resuming training from step {RESUME_FROM_STEP} and targeting total {NEW_TOTAL_MAX_STEPS} steps.")
print(f"    Targeting a WER below {WER_TARGET:.2f}%. Early stopping patience: {EARLY_STOPPING_PATIENCE} evaluations ({EARLY_STOPPING_PATIENCE * EVAL_STEPS} steps).")
print(f"{'='*60}")

# --- FIX: UnpicklingError when resuming training ---
# This is required for torch.load to successfully load the random number generator state
# from the checkpoint when weights_only=True is the default (PyTorch 2.6+).
# Add specific numpy globals that might be present in the checkpoint.
torch.serialization.add_safe_globals([np.dtypes.UInt32DType, np._core.multiarray._reconstruct, np.ndarray, np.dtype])
# -----------------------------------------------------------------------------

# Assume processor, train_ds, eval_ds, compute_metrics are available from previous cells
# Re-initialize DataCollator
whisper_cfg = WhisperConfig.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
decoder_start_id = whisper_cfg.decoder_start_token_id
del whisper_cfg

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=decoder_start_id
)

# Create new training arguments with the updated max_steps
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=NEW_TOTAL_MAX_STEPS, # Use the new total max steps
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    optim="adamw_torch_fused",
    lr_scheduler_type="linear",
    fp16=FP16,
    evaluation_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=225,
    logging_steps=25,
    report_to=["none"],
    push_to_hub=False,
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE, early_stopping_threshold=EARLY_STOPPING_THRESHOLD)]
)

# Call trainer.train() with the explicitly specified checkpoint
trainer.train(resume_from_checkpoint=resume_from_checkpoint_path)

# Evaluate after the training to see the final WER
results = trainer.evaluate()
final_wer = results.get("eval_wer", 999.0)

print(f"\n{'='*60}")
if final_wer <= WER_TARGET:
    print(f"    Training completed! Final WER: {final_wer:.2f}% (achieved target below {WER_TARGET:.2f}%)")
elif trainer.state.global_step < NEW_TOTAL_MAX_STEPS:
    print(f"    Training stopped early due to no improvement in WER for {EARLY_STOPPING_PATIENCE * EVAL_STEPS} steps. Final WER: {final_wer:.2f}%. Target below {WER_TARGET:.2f}% not yet met.")
    print("    Suggestions: Consider reviewing hyperparameters (learning rate, batch size, LoRA config) or increasing the dataset size.")
else:
    print(f"    Training completed up to {NEW_TOTAL_MAX_STEPS} steps. Final WER: {final_wer:.2f}%. Target below {WER_TARGET:.2f}% not met.")
print(f"  Checkpoints are saved in: {OUTPUT_DIR}")
print(f"{'='*60}")

# Free up memory
del trainer, training_args
gc.collect()
torch.cuda.empty_cache()

   Re-injected LoRA into 72 projection layers with current LORA_R=128
   Trainable : 14,155,776  /  Total : 255,890,688  (5.53%)
 Model ready (plain Whisper + manual LoRA — no PEFT wrapper)

    Resuming training from step 7000 and targeting total 8000 steps.
    Targeting a WER below 39.99%. Early stopping patience: 3 evaluations (600 steps).


max_steps is given, it will override any value given in num_train_epochs
There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Step,Training Loss,Validation Loss,Wer
7200,0.683000,0.547837,43.830000
7400,0.702600,0.549081,44.270000
7600,0.700000,0.550577,44.120000
7800,0.675500,0.549273,43.860000


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-cust


    Training stopped early due to no improvement in WER for 600 steps. Final WER: 43.83%. Target below 39.99% not yet met.
    Suggestions: Consider reviewing hyperparameters (learning rate, batch size, LoRA config) or increasing the dataset size.
  Checkpoints are saved in: /content/drive/MyDrive/Audios_Phusto/whisper-pashto-lora


In [ ]:
import os
from transformers import WhisperProcessor # Added import

# Re-initialize processor if it's not defined
if 'processor' not in locals() and 'processor' not in globals():
    print("WARNING: 'processor' not found, re-initializing from pre-trained model.")
    processor = WhisperProcessor.from_pretrained(
        MODEL_NAME, language=LANGUAGE, task=TASK, cache_dir=CACHE_DIR
    )

# ── Save Best WER Model (full model with LoRA) and Processor ─────────────────
# This saves the complete model (base + fine-tuned LoRA weights) and the processor
# in a format suitable for Hugging Face Hub upload or direct inference.
model_save_dir_local = os.path.join(OUTPUT_DIR, "best_wer_model_for_hf")
os.makedirs(model_save_dir_local, exist_ok=True)
model.save_pretrained(model_save_dir_local)
processor.save_pretrained(model_save_dir_local)
print(f" Best WER model and processor saved locally to: {model_save_dir_local}")

# Also save the original lora_adapter format for backward compatibility or specific LoRA loading
lora_adapter_dir = os.path.join(OUTPUT_DIR, "lora_adapter")
os.makedirs(lora_adapter_dir, exist_ok=True)
model.save_pretrained(lora_adapter_dir) # This saves the full model, which includes LoRA
processor.save_pretrained(lora_adapter_dir)
print(f" LoRA adapter directory updated locally to: {lora_adapter_dir}")

# ── Copy to Drive so it survives Colab disconnection ─────────────────────────
model_save_dir_drive = "/content/drive/MyDrive/whisper_pashto_best_wer_model"
lora_adapter_drive = "/content/drive/MyDrive/whisper_pashto_lora_adapter"
import shutil

# Copy the complete best WER model for easy download/HF push
shutil.copytree(model_save_dir_local, model_save_dir_drive, dirs_exist_ok=True)
print(f" Also copied complete best WER model to Drive: {model_save_dir_drive}")

# Copy the lora_adapter directory as well for continued use/evaluation
shutil.copytree(lora_adapter_dir, lora_adapter_drive, dirs_exist_ok=True)
print(f" Also copied lora_adapter to Drive: {lora_adapter_drive}")

print("\n--- Instructions for Hugging Face Hub Upload ---")
print(f"You can now upload the model from your Google Drive: {model_save_dir_drive}")
print("To do so, you can use the Hugging Face `push_to_hub` functionality or manually upload the contents of this folder to a new model repository on huggingface.co")

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}


 Best WER model and processor saved locally to: /content/drive/MyDrive/Audios_Phusto/whisper-pashto-lora/best_wer_model_for_hf
 LoRA adapter directory updated locally to: /content/drive/MyDrive/Audios_Phusto/whisper-pashto-lora/lora_adapter
 Also copied complete best WER model to Drive: /content/drive/MyDrive/whisper_pashto_best_wer_model
 Also copied lora_adapter to Drive: /content/drive/MyDrive/whisper_pashto_lora_adapter

--- Instructions for Hugging Face Hub Upload ---
You can now upload the model from your Google Drive: /content/drive/MyDrive/whisper_pashto_best_wer_model
To do so, you can use the Hugging Face `push_to_hub` functionality or manually upload the contents of this folder to a new model repository on huggingface.co


In [ ]:
print(f"Contents of lora_adapter_dir ({lora_adapter_dir}):")
!ls -F "{lora_adapter_dir}"

Contents of lora_adapter_dir (/content/drive/MyDrive/Audios_Phusto/whisper-pashto-lora/lora_adapter):
added_tokens.json	merges.txt	   preprocessor_config.json  vocab.json
config.json		model.safetensors  special_tokens_map.json
generation_config.json	normalizer.json    tokenizer_config.json


In [ ]:
import os
import torch
import torch.nn as nn
import safetensors.torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor

# Re-define LoRALinear class (copied from previous cells for self-containment)
class LoRALinear(nn.Module):
    """Drop-in LoRA wrapper for nn.Linear. Adds A*B low-rank branch."""
    def __init__(self, linear: nn.Linear, r: int, alpha: int, dropout: float):
        super().__init__()
        self.linear   = linear          # frozen original weight
        self.r        = r
        self.scaling  = alpha / r
        self.lora_A   = nn.Linear(linear.in_features,  r, bias=False)
        self.lora_B   = nn.Linear(r, linear.out_features, bias=False)
        self.dropout  = nn.Dropout(dropout)
        # initialise: A ~ N(0,1), B = 0  → LoRA starts as identity delta
        nn.init.normal_(self.lora_A.weight)
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        return self.linear(x) + self.lora_B(self.lora_A(self.dropout(x))) * self.scaling

# Re-define inject_lora function (copied from previous cells for self-containment)
def inject_lora(model, r, alpha, dropout, target_suffixes=("q_proj", "v_proj")):
    """Replace matching Linear layers with LoRALinear in-place."""
    modules_to_replace = []
    for name, module in model.named_modules():
        for suffix in target_suffixes:
            if name.endswith(suffix) and isinstance(module, nn.Linear):
                parts = name.split(".")
                parent_name = ".".join(parts[:-1])
                parent_module = model.get_submodule(parent_name) if parent_name else model
                modules_to_replace.append((parent_module, parts[-1], module))

    for parent, child_name, original_linear in modules_to_replace:
        setattr(parent, child_name, LoRALinear(original_linear, r=r, alpha=alpha, dropout=dropout))
    return len(modules_to_replace)

def merge_lora_layers(model):
    """
    Merges LoRA weights back into the base linear layers and replaces LoRALinear
    modules with standard nn.Linear modules.
    """
    modules_to_merge = []
    for name, module in model.named_modules():
        if isinstance(module, LoRALinear):
            # Collect (parent_module, child_attribute_name, LoRALinear_instance)
            parts = name.split(".")
            parent_name = ".".join(parts[:-1])
            parent_module = model.get_submodule(parent_name) if parent_name else model
            modules_to_merge.append((parent_module, parts[-1], module))

    for parent, child_name, lora_module in modules_to_merge:
        # Calculate the delta weight from LoRA layers
        delta_weight = (lora_module.lora_B.weight @ lora_module.lora_A.weight) * lora_module.scaling

        # Apply delta to the original linear layer's weight
        original_linear = lora_module.linear
        # Ensure weights are on the same device before addition
        original_linear.weight.data += delta_weight.to(original_linear.weight.device)

        # Replace the LoRALinear module with the modified original nn.Linear
        setattr(parent, child_name, original_linear)
        del lora_module # Free up memory associated with the LoRA wrapper

# Configuration from previous cells
# Ensure these variables are defined in your notebook's global scope or explicitly set here
MODEL_NAME = "openai/whisper-small" # e.g., from cell iVoZ7BxzkJH4
CACHE_DIR = "/content/cache" # e.g., from cell iVoZ7BxzkJH4
LORA_R = 128 # e.g., from cell iVoZ7BxzkJH4
LORA_ALPHA = 256 # e.g., from cell iVoZ7BxzkJH4
LORA_DROPOUT = 0.15 # e.g., from cell iVoZ7BxzkJH4
OUTPUT_DIR = "/content/drive/MyDrive/Audios_Phusto/whisper-pashto-lora" # e.g., from cell iVoZ7BxzkJH4

# Path where the LoRA model was saved (containing model.safetensors and processor files)
LORA_ADAPTER_SAVED_PATH = os.path.join(OUTPUT_DIR, "lora_adapter")
# New path for the fully merged model for easy inference loading
MERGED_MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, "merged_whisper_lora_pashto")

print(f"Loading base model: {MODEL_NAME}")
# 1. Load the base Whisper model
base_model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME, cache_dir=CACHE_DIR
)

# 2. Inject the custom LoRA layers into the base model
print(f"Injecting LoRA layers with r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
n_injected = inject_lora(base_model, r=LORA_R, alpha=LORA_ALPHA, dropout=LORA_DROPOUT)
print(f"  Injected {n_injected} LoRA modules.")

# 3. Load the saved LoRA adapter weights into the model with LoRA layers injected
print(f"Loading LoRA adapter weights from: {LORA_ADAPTER_SAVED_PATH}/model.safetensors")
lora_state_dict = safetensors.torch.load_file(os.path.join(LORA_ADAPTER_SAVED_PATH, "model.safetensors"))

# Load the state dictionary. strict=False allows for keys in lora_state_dict that are not in base_model
# (e.g. if the original linear weights were not saved, but LoRA A and B are.)
base_model.load_state_dict(lora_state_dict, strict=False)
print("  LoRA adapter weights loaded into the model.")

# 4. Merge the LoRA weights into the base model's linear layers
print("Merging LoRA weights into base model layers...")
merge_lora_layers(base_model)
print("  LoRA weights merged successfully.")

# 5. Save the fully merged model and processor
print(f"Saving fully merged model to: {MERGED_MODEL_SAVE_PATH}")
os.makedirs(MERGED_MODEL_SAVE_PATH, exist_ok=True)
base_model.save_pretrained(MERGED_MODEL_SAVE_PATH)

# Save the processor as well, from the original lora_adapter path, to the merged model path
processor = WhisperProcessor.from_pretrained(LORA_ADAPTER_SAVED_PATH)
processor.save_pretrained(MERGED_MODEL_SAVE_PATH)
print("  Merged model and processor saved.")

# 6. Demonstrate loading the merged model for inference
print("\nDemonstrating inference loading from the merged model path:")
inference_model = WhisperForConditionalGeneration.from_pretrained(MERGED_MODEL_SAVE_PATH)
inference_processor = WhisperProcessor.from_pretrained(MERGED_MODEL_SAVE_PATH)

inference_model.eval()
print("  Merged model loaded successfully for inference using from_pretrained!")
print("You can now use 'inference_model' and 'inference_processor' directly.")
print(f"\nTo use this merged model in your Streamlit app, update the ADAPTER_PATH and PROCESSOR_PATH variables in app.py to point to: {MERGED_MODEL_SAVE_PATH}")


Loading base model: openai/whisper-small


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

...

  LoRA weights merged successfully.
Saving fully merged model to: /content/drive/MyDrive/Audios_Phusto/whisper-pashto-lora/merged_whisper_lora_pashto


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


  Merged model and processor saved.

Demonstrating inference loading from the merged model path:


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


  Merged model loaded successfully for inference using from_pretrained!
You can now use 'inference_model' and 'inference_processor' directly.

To use this merged model in your Streamlit app, update the ADAPTER_PATH and PROCESSOR_PATH variables in app.py to point to: /content/drive/MyDrive/Audios_Phusto/whisper-pashto-lora/merged_whisper_lora_pashto


### Loading the fine-tuned model with custom LoRA for inference

Since we manually injected LoRA layers and saved the entire model, we need a special loading procedure to ensure the LoRA weights are properly recognized.

In [3]:
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import os

# Define the path to the fully merged model (this path is available from previous steps)
MERGED_MODEL_SAVE_PATH = "/content/drive/MyDrive/Audios_Phusto/whisper-pashto-lora/merged_whisper_lora_pashto"

# Load the fully merged Whisper model for inference
print(f"Loading fully merged model from: {MERGED_MODEL_SAVE_PATH}")
model_for_inference = WhisperForConditionalGeneration.from_pretrained(MERGED_MODEL_SAVE_PATH)

# Load the processor (tokenizer and feature extractor) from the merged model path
inference_processor = WhisperProcessor.from_pretrained(MERGED_MODEL_SAVE_PATH)

# Set model to evaluation mode
model_for_inference.eval()

print(" Fully merged model loaded successfully for inference")

Loading fully merged model from: /content/drive/MyDrive/Audios_Phusto/whisper-pashto-lora/merged_whisper_lora_pashto


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


 Fully merged model loaded successfully for inference


## Streamlit Application Setup (Part 1: Upload and Transcribe)

First, we need to install Streamlit and a tool to expose the web application. We'll then create the `app.py` file, which will contain all the logic for your Streamlit application. This file will load your fine-tuned model and allow users to upload audio files for transcription.

In [4]:
!pip install streamlit pyngrok pydub
print(" Streamlit and dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 121.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 113.4 MB/s eta 0:00:00
 Streamlit and dependencies installed


In [8]:
%%writefile app.py
import streamlit as st
import torch
import librosa
import os
import sqlite3
import hashlib
import csv
import io
import requests
import json
from datetime import datetime
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from transformers import pipeline as hf_pipeline
from difflib import SequenceMatcher

try:
    from jiwer import wer as jiwer_wer
    JIWER_AVAILABLE = True
except ImportError:
    JIWER_AVAILABLE = False

# ── Config ────────────────────────────────────────────────────────────────────
# Merged model path — LoRA weights already baked in, no injection needed
MERGED_MODEL_PATH = os.environ.get(
    "MERGED_MODEL_PATH",
    "/content/drive/MyDrive/Audios_Phusto/whisper-pashto-lora/merged_whisper_lora_pashto"
)
FP16_ENABLED = os.environ.get("FP16", "False").lower() == "true"

DRIVE_BASE  = "/content/drive/MyDrive/Audios_Phusto/whisper-pashto-lora"
DB_PATH     = os.path.join(DRIVE_BASE, "pashto_app.db")
AUDIO_STORE = os.path.join(DRIVE_BASE, "audio_uploads")
os.makedirs(AUDIO_STORE, exist_ok=True)

SUPPORTED_FORMATS = ["wav", "mp3", "mp4", "m4a", "flac", "ogg", "opus", "webm", "aac", "wma"]

NLLB_MODEL   = "facebook/nllb-200-distilled-600M"
PASHTO_CODE  = "pbt_Arab"
ENGLISH_CODE = "eng_Latn"
URDU_CODE    = "urd_Arab"

# ── AI Config ─────────────────────────────────────────────────────────────────
HF_TOKEN         = os.environ.get("HF_TOKEN", "")
HF_INFERENCE_URL = "https://api-inference.huggingface.co/models/"

# ── Database ──────────────────────────────────────────────────────────────────

def get_db():
    return sqlite3.connect(DB_PATH, check_same_thread=False)


def init_db():
    conn = get_db()
    c = conn.cursor()
    c.execute(
        "CREATE TABLE IF NOT EXISTS users ("
        "id INTEGER PRIMARY KEY AUTOINCREMENT,"
        "username TEXT UNIQUE NOT NULL,"
        "password_hash TEXT NOT NULL,"
        "email TEXT,"
        "created_at TEXT)"
    )
    c.execute(
        "CREATE TABLE IF NOT EXISTS transcriptions ("
        "id INTEGER PRIMARY KEY AUTOINCREMENT,"
        "user_id INTEGER NOT NULL,"
        "filename TEXT NOT NULL,"
        "original_transcription TEXT,"
        "edited_transcription TEXT,"
        "reference_text TEXT,"
        "wer_score REAL,"
        "translation_en TEXT,"
        "translation_ur TEXT,"
        "audio_filepath TEXT,"
        "created_at TEXT,"
        "FOREIGN KEY(user_id) REFERENCES users(id))"
    )
    conn.commit()
    existing_columns = {row[1] for row in c.execute("PRAGMA table_info(transcriptions)")}
    migrations = {
        "translation_en":       "ALTER TABLE transcriptions ADD COLUMN translation_en TEXT",
        "translation_ur":       "ALTER TABLE transcriptions ADD COLUMN translation_ur TEXT",
        "wer_score":            "ALTER TABLE transcriptions ADD COLUMN wer_score REAL",
        "reference_text":       "ALTER TABLE transcriptions ADD COLUMN reference_text TEXT",
        "edited_transcription": "ALTER TABLE transcriptions ADD COLUMN edited_transcription TEXT",
        "audio_filepath":       "ALTER TABLE transcriptions ADD COLUMN audio_filepath TEXT",
    }
    for col, sql in migrations.items():
        if col not in existing_columns:
            c.execute(sql)
    conn.commit()
    conn.close()


init_db()


def _hash(pw):
    return hashlib.sha256(pw.encode()).hexdigest()


def create_user(username, password, email=""):
    conn = get_db()
    try:
        conn.execute(
            "INSERT INTO users (username, password_hash, email, created_at) VALUES (?, ?, ?, ?)",
            (username, _hash(password), email, datetime.now().isoformat())
        )
        conn.commit()
        return True, "Account created! Please log in."
    except sqlite3.IntegrityError:
        return False, "Username already exists."
    finally:
        conn.close()


def verify_user(username, password):
    conn = get_db()
    row = conn.execute(
        "SELECT id, password_hash FROM users WHERE username = ?", (username,)
    ).fetchone()
    conn.close()
    if row and row[1] == _hash(password):
        return True, row[0]
    return False, None


def save_transcription(user_id, filename, original, audio_filepath=None, reference=None, wer_score=None):
    conn = get_db()
    c = conn.cursor()
    existing = c.execute(
        "SELECT id FROM transcriptions "
        "WHERE user_id=? AND filename=? AND original_transcription=? AND (audio_filepath=? OR audio_filepath IS NULL)",
        (user_id, filename, original, audio_filepath)
    ).fetchone()
    if existing:
        conn.close()
        return existing[0]
    c.execute(
        "INSERT INTO transcriptions "
        "(user_id, filename, original_transcription, edited_transcription, "
        "reference_text, wer_score, translation_en, translation_ur, audio_filepath, created_at) "
        "VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
        (user_id, filename, original, original, reference, wer_score,
         None, None, audio_filepath, datetime.now().isoformat())
    )
    conn.commit()
    tid = c.lastrowid
    conn.close()
    return tid


def update_transcription(tid, edited, reference=None, wer_score=None):
    conn = get_db()
    conn.execute(
        "UPDATE transcriptions SET edited_transcription=?, reference_text=?, wer_score=? WHERE id=?",
        (edited, reference, wer_score, tid)
    )
    conn.commit()
    conn.close()


def delete_transcription(tid, audio_filepath=None):
    conn = get_db()
    conn.execute("DELETE FROM transcriptions WHERE id=?", (tid,))
    conn.commit()
    conn.close()
    if audio_filepath and os.path.exists(audio_filepath):
        try:
            os.remove(audio_filepath)
            parent_dir = os.path.dirname(audio_filepath)
            if parent_dir != AUDIO_STORE and not os.listdir(parent_dir):
                os.rmdir(parent_dir)
        except OSError as e:
            st.warning(f"Could not delete audio file: {e}")


def save_translations(tid, en_text, ur_text):
    conn = get_db()
    conn.execute(
        "UPDATE transcriptions SET translation_en=?, translation_ur=? WHERE id=?",
        (en_text, ur_text, tid)
    )
    conn.commit()
    conn.close()


def get_user_history(user_id):
    conn = get_db()
    rows = conn.execute(
        "SELECT id, filename, original_transcription, edited_transcription, "
        "reference_text, wer_score, translation_en, translation_ur, audio_filepath, created_at "
        "FROM transcriptions WHERE user_id=? ORDER BY created_at DESC",
        (user_id,)
    ).fetchall()
    conn.close()
    return rows

# ── Whisper Model — Merged (no LoRA injection needed) ─────────────────────────

@st.cache_resource
def load_whisper_model():
    """
    Loads the fully merged Whisper+LoRA model using standard from_pretrained.
    No LoRA injection, no safetensors manual loading — works like any HF model.
    """
    if not os.path.exists(MERGED_MODEL_PATH):
        raise FileNotFoundError(
            f"Merged model not found at:\n{MERGED_MODEL_PATH}\n\n"
            "Run the merge cell in your notebook first, then restart the app."
        )

    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = WhisperForConditionalGeneration.from_pretrained(MERGED_MODEL_PATH)
    model = model.to(device)

    if FP16_ENABLED and device == "cuda":
        model = model.half()

    model.eval()

    processor = WhisperProcessor.from_pretrained(MERGED_MODEL_PATH)

    return model, processor

# ── Translation — NLLB-200 600M ───────────────────────────────────────────────

@st.cache_resource
def load_translation_model():
    return hf_pipeline(
        "translation",
        model=NLLB_MODEL,
        device=-1,
        torch_dtype=torch.float32,
    )


def translate_text(text, target_lang):
    if not text or not text.strip():
        return ""
    try:
        translator = load_translation_model()
        result = translator(
            text,
            src_lang=PASHTO_CODE,
            tgt_lang=target_lang,
            max_length=512,
            num_beams=4,
        )
        return result[0]["translation_text"]
    except Exception as e:
        return "Translation error: " + str(e)


def verify_translation(original_pashto, translated_text, target_lang):
    """
    Back-translates output → Pashto and measures character similarity.
    Returns (score_percent, label, back_translated_text)
    """
    if not translated_text or "error" in translated_text.lower():
        return None, "Could not verify", ""
    try:
        translator = load_translation_model()
        back = translator(
            translated_text,
            src_lang=target_lang,
            tgt_lang=PASHTO_CODE,
            max_length=512,
            num_beams=4,
        )
        back_text = back[0]["translation_text"]
        ratio = SequenceMatcher(None, original_pashto.strip(), back_text.strip()).ratio()
        score = round(ratio * 100, 1)
        if score >= 75:
            label = "✅ High confidence"
        elif score >= 50:
            label = "⚠️ Medium confidence"
        else:
            label = "❌ Low confidence — review recommended"
        return score, label, back_text
    except Exception as e:
        return None, f"Verification error: {e}", ""

# ── Transcription ─────────────────────────────────────────────────────────────

def transcribe_audio(audio_path):
    model, processor = load_whisper_model()
    audio, sr = librosa.load(audio_path, sr=16000, mono=True)
    features = processor.feature_extractor(
        audio, sampling_rate=sr, return_tensors="pt"
    ).input_features
    device = "cuda" if torch.cuda.is_available() else "cpu"
    with torch.no_grad():
        if FP16_ENABLED and device == "cuda":
            features = features.half()
        ids = model.generate(
            features.to(device),
            language="pashto",
            task="transcribe"
        )
    return processor.tokenizer.batch_decode(ids, skip_special_tokens=True)[0]


def compute_wer(reference, hypothesis):
    if not JIWER_AVAILABLE or not reference.strip():
        return None
    try:
        score = jiwer_wer(reference.strip(), hypothesis.strip())
        return round(score * 100, 2)
    except Exception:
        return None

# ── Export helpers ────────────────────────────────────────────────────────────

def make_csv_bytes(rows, headers):
    """UTF-8 BOM ensures Excel opens Pashto/Urdu text correctly."""
    buf = io.StringIO()
    w = csv.writer(buf)
    w.writerow(headers)
    for row in rows:
        w.writerow([str(cell) if cell is not None else "" for cell in row])
    return ("\ufeff" + buf.getvalue()).encode("utf-8")


def make_excel_bytes(rows, headers):
    """Returns .xlsx with right-aligned Pashto/Urdu columns."""
    try:
        import openpyxl
        from openpyxl.styles import Font, Alignment
        wb = openpyxl.Workbook()
        ws = wb.active
        ws.title = "Transcriptions"
        for col_idx, header in enumerate(headers, 1):
            cell = ws.cell(row=1, column=col_idx, value=header)
            cell.font = Font(bold=True)
            cell.alignment = Alignment(horizontal="center")
        rtl_cols = {"Pashto", "Urdu", "Original", "Edited", "Reference"}
        for row_idx, row in enumerate(rows, 2):
            for col_idx, (header, cell_val) in enumerate(zip(headers, row), 1):
                val = str(cell_val) if cell_val is not None else ""
                cell = ws.cell(row=row_idx, column=col_idx, value=val)
                if any(rtl in header for rtl in rtl_cols):
                    cell.alignment = Alignment(horizontal="right")
        buf = io.BytesIO()
        wb.save(buf)
        buf.seek(0)
        return buf.read()
    except ImportError:
        return make_csv_bytes(rows, headers)

# ── AI Chat — 3-strategy free approach ───────────────────────────────────────

def _try_hf_inference(prompt, system_msg):
    if not HF_TOKEN:
        return None
    req_headers = {
        "Authorization": f"Bearer {HF_TOKEN}",
        "Content-Type": "application/json",
    }
    models_to_try = [
        ("HuggingFaceH4/zephyr-7b-beta",
         f"<|system|>\n{system_msg}</s>\n<|user|>\n{prompt}</s>\n<|assistant|>"),
        ("mistralai/Mistral-7B-Instruct-v0.1",
         f"[INST] {system_msg}\n\n{prompt} [/INST]"),
        ("google/flan-t5-large",
         f"{system_msg}\n\n{prompt}"),
        ("gpt2-medium",
         f"{system_msg}\n\n{prompt}"),
    ]
    for model_id, formatted_prompt in models_to_try:
        url = f"https://api-inference.huggingface.co/models/{model_id}"
        payload = {
            "inputs": formatted_prompt,
            "parameters": {"max_new_tokens": 400, "temperature": 0.7,
                           "do_sample": True, "return_full_text": False},
            "options": {"wait_for_model": True, "use_cache": False}
        }
        try:
            r = requests.post(url, headers=req_headers, json=payload, timeout=90)
            if r.status_code == 200:
                data = r.json()
                text = ""
                if isinstance(data, list) and data:
                    text = data[0].get("generated_text", "").strip()
                elif isinstance(data, dict):
                    text = data.get("generated_text", "").strip()
                if text and len(text) > 10:
                    return text
        except Exception:
            continue
    return None


def _try_openai_compatible(prompt, system_msg):
    if not HF_TOKEN:
        return None
    try:
        url = "https://api-inference.huggingface.co/v1/chat/completions"
        req_headers = {"Authorization": f"Bearer {HF_TOKEN}", "Content-Type": "application/json"}
        payload = {
            "model": "HuggingFaceH4/zephyr-7b-beta",
            "messages": [{"role": "system", "content": system_msg},
                         {"role": "user",   "content": prompt}],
            "max_tokens": 400, "temperature": 0.7,
        }
        r = requests.post(url, headers=req_headers, json=payload, timeout=60)
        if r.status_code == 200:
            text = r.json()["choices"][0]["message"]["content"].strip()
            if text:
                return text
    except Exception:
        pass
    return None


def _try_pollinations(prompt, system_msg):
    try:
        import urllib.parse
        combined = f"{system_msg}\n\nUser question: {prompt}"
        encoded  = urllib.parse.quote(combined[:800])
        r = requests.get(f"https://text.pollinations.ai/{encoded}", timeout=60)
        if r.status_code == 200 and r.text.strip():
            return r.text.strip()
    except Exception:
        pass
    return None


def ask_ai_free(prompt, system_msg):
    result = _try_hf_inference(prompt, system_msg)
    if result:
        return result, None
    result = _try_openai_compatible(prompt, system_msg)
    if result:
        return result, None
    result = _try_pollinations(prompt, system_msg)
    if result:
        return result, None
    return None, (
        "All AI strategies failed.\n\n"
        "1. **Set HF_TOKEN** — `os.environ['HF_TOKEN'] = 'hf_...'` then restart\n"
        "2. **Wait 30 seconds** and try again (free models may be waking up)\n"
        "3. **Check internet** — Colab needs network access for AI"
    )

# ── UI Helpers ────────────────────────────────────────────────────────────────

def wer_badge(wer_score):
    if wer_score is None:
        return
    if wer_score <= 15:
        bg, label = "#22c55e", "Excellent"
    elif wer_score <= 35:
        bg, label = "#f59e0b", "Good"
    elif wer_score <= 60:
        bg, label = "#f97316", "Fair"
    else:
        bg, label = "#ef4444", "Poor"
    accuracy = max(0, round(100 - wer_score, 1))
    st.markdown(
        f'<div style="display:flex;gap:10px;align-items:center;margin:8px 0;">'
        f'<div style="background:{bg};color:#fff;font-weight:700;padding:5px 14px;border-radius:20px;font-size:0.85rem;">'
        f'WER: {wer_score}%</div>'
        f'<div style="background:#1e293b;color:#94a3b8;font-size:0.82rem;padding:5px 12px;border-radius:20px;">'
        f'~{accuracy}% accuracy — {label}</div></div>',
        unsafe_allow_html=True
    )


def show_translation_box(tid, pashto_text, existing_en, existing_ur):
    st.markdown("**Translate:**")
    col_en, col_ur, col_both = st.columns(3)

    with col_en:
        if st.button("To English", key="en_" + str(tid), use_container_width=True):
            with st.spinner("Translating to English..."):
                en = translate_text(pashto_text, ENGLISH_CODE)
            save_translations(tid, en, existing_ur)
            st.session_state["t_en_" + str(tid)] = en
            st.rerun()
    with col_ur:
        if st.button("To Urdu", key="ur_" + str(tid), use_container_width=True):
            with st.spinner("Translating to Urdu..."):
                ur = translate_text(pashto_text, URDU_CODE)
            save_translations(tid, existing_en, ur)
            st.session_state["t_ur_" + str(tid)] = ur
            st.rerun()
    with col_both:
        if st.button("Both Languages", key="both_" + str(tid), use_container_width=True):
            with st.spinner("Translating..."):
                en = translate_text(pashto_text, ENGLISH_CODE)
                ur = translate_text(pashto_text, URDU_CODE)
            save_translations(tid, en, ur)
            st.session_state["t_en_" + str(tid)] = en
            st.session_state["t_ur_" + str(tid)] = ur
            st.rerun()

    en_val = st.session_state.get("t_en_" + str(tid), existing_en or "")
    ur_val = st.session_state.get("t_ur_" + str(tid), existing_ur or "")

    if en_val:
        st.text_area("English", value=en_val, height=85, key="disp_en_" + str(tid))
        col_dl_en, col_vfy_en = st.columns(2)
        with col_dl_en:
            st.download_button("Download English", data=en_val.encode("utf-8"),
                               file_name="en_" + str(tid) + ".txt",
                               mime="text/plain; charset=utf-8", key="dl_en_" + str(tid))
        with col_vfy_en:
            # Button key uses "btn_" prefix — different from session state key "vfy_en_result_"
            if st.button("Verify English", key="btn_vfy_en_" + str(tid), use_container_width=True):
                with st.spinner("Back-translating to verify..."):
                    score, lbl, back = verify_translation(pashto_text, en_val, ENGLISH_CODE)
                # Store result under a different key than the button widget
                st.session_state["vfy_en_result_" + str(tid)] = (score, lbl, back)
        # Read result from session state using the result key (not the button key)
        vfy = st.session_state.get("vfy_en_result_" + str(tid))
        if vfy:
            score, lbl, back = vfy
            st.info(f"**Translation Quality:** {lbl} (similarity {score}%)")
            if back:
                with st.expander("Back-translated Pashto"):
                    st.text(back)

    if ur_val:
        st.text_area("Urdu", value=ur_val, height=85, key="disp_ur_" + str(tid))
        col_dl_ur, col_vfy_ur = st.columns(2)
        with col_dl_ur:
            st.download_button("Download Urdu", data=ur_val.encode("utf-8"),
                               file_name="ur_" + str(tid) + ".txt",
                               mime="text/plain; charset=utf-8", key="dl_ur_" + str(tid))
        with col_vfy_ur:
            # Button key uses "btn_" prefix — different from session state key "vfy_ur_result_"
            if st.button("Verify Urdu", key="btn_vfy_ur_" + str(tid), use_container_width=True):
                with st.spinner("Back-translating to verify..."):
                    score, lbl, back = verify_translation(pashto_text, ur_val, URDU_CODE)
                # Store result under a different key than the button widget
                st.session_state["vfy_ur_result_" + str(tid)] = (score, lbl, back)
        # Read result from session state using the result key (not the button key)
        vfy = st.session_state.get("vfy_ur_result_" + str(tid))
        if vfy:
            score, lbl, back = vfy
            st.info(f"**Translation Quality:** {lbl} (similarity {score}%)")
            if back:
                with st.expander("Back-translated Pashto"):
                    st.text(back)

# ── Pages ─────────────────────────────────────────────────────────────────────

def about_page():
    st.markdown("""
    <style>
    .about-hero{background:linear-gradient(135deg,#1e1b4b,#312e81);border-radius:16px;
        padding:2.5rem 2rem;margin-bottom:1.5rem;color:white;}
    .about-hero h1{font-size:2.2rem;font-weight:800;margin-bottom:0.3rem;}
    .about-hero p{font-size:1.05rem;color:#c7d2fe;}
    .feature-card{background:#1e293b;border:1px solid #334155;border-radius:12px;
        padding:1.2rem 1.4rem;margin-bottom:0.8rem;color:#e2e8f0;}
    .feature-card h4{color:#a78bfa;margin-bottom:0.3rem;font-size:1rem;}
    .feature-card p{font-size:0.88rem;color:#94a3b8;margin:0;}
    .step-box{background:#0f172a;border-left:4px solid #6366f1;padding:0.8rem 1rem;
        border-radius:0 8px 8px 0;margin-bottom:0.6rem;color:#e2e8f0;font-size:0.9rem;}
    </style>
    <div class="about-hero">
        <h1>🎙️ Pashto Whisper STT</h1>
        <p>A fine-tuned Speech-to-Text system for Pakistani Pashto — built for agriculture,
        health, food, services domains.</p>
    </div>
    """, unsafe_allow_html=True)

    st.subheader("What is this app?")
    st.markdown("""
    This application uses a **fine-tuned OpenAI Whisper** model trained specifically on
    **Pakistani Pashto** speech. LoRA weights are fully merged into the base model —
    so it loads like any standard Whisper model with no extra steps.
    It converts spoken Pashto into written Pashto text, then optionally translates to English or Urdu.
    """)
    st.markdown("---")

    st.subheader("Features")
    features = [
        ("🎤 Multi-format Audio Upload",   "WAV, MP3, M4A, OGG, FLAC, MP4, OPUS, WEBM, AAC, WMA."),
        ("📝 Pashto Transcription",        "Merged Whisper model — faster loading, same accuracy."),
        ("✏️ Editable Transcriptions",     "Correct mistakes and save the improved version."),
        ("📊 WER Scoring",                 "Paste reference text to get Word Error Rate + accuracy %."),
        ("🌐 English & Urdu Translation",  "NLLB-200 600M runs locally — no API needed."),
        ("✅ Translation Verification",    "Back-translates output and shows similarity score."),
        ("📁 CSV & Excel Export",          "UTF-8 BOM encoding — Pashto/Urdu text displays correctly."),
        ("🤖 AI Assistant",                "Three-strategy free AI — works without paid subscriptions."),
        ("🔒 User Accounts",               "Private accounts — your data stored separately."),
        ("📂 Audio Playback in History",   "Replay any uploaded audio directly from history."),
    ]
    col1, col2 = st.columns(2)
    for i, (title, desc) in enumerate(features):
        target = col1 if i % 2 == 0 else col2
        with target:
            st.markdown(
                f'<div class="feature-card"><h4>{title}</h4><p>{desc}</p></div>',
                unsafe_allow_html=True
            )
    st.markdown("---")

    st.subheader("How to Use")
    steps = [
        ("1️⃣ Create an account",         "Click 'Create Account', set username + password, then log in."),
        ("2️⃣ Go to Transcribe",           "Select Transcribe from the sidebar."),
        ("3️⃣ Upload audio",               "Select one or more Pashto audio files in any format."),
        ("4️⃣ Add reference (optional)",   "Paste correct Pashto text to get WER score."),
        ("5️⃣ Click Transcribe All",       "Model processes files and shows Pashto transcripts."),
        ("6️⃣ Review in History",          "Go to History to edit, translate, or export."),
        ("7️⃣ Translate",                  "Click 'To English' or 'To Urdu' in any history record."),
        ("8️⃣ Verify translation",         "Click Verify to check quality via back-translation."),
        ("9️⃣ Use AI Assistant",           "Select a transcription and ask the AI anything about it."),
        ("🔟 Export",                      "Download as CSV or Excel — Arabic script displays correctly."),
    ]
    for title, desc in steps:
        st.markdown(
            f'<div class="step-box"><strong>{title}</strong> — {desc}</div>',
            unsafe_allow_html=True
        )
    st.markdown("---")

    st.subheader("Technical Details")
    col_a, col_b = st.columns(2)
    with col_a:
        st.markdown("""
**Speech Model**
- Base: OpenAI Whisper Small
- Fine-tuning: LoRA → fully merged
- Language: Pakistani Pashto
- Training: 5,000+ audio clips

**Domains covered**
- Agriculture · Health · Food
- Services · General Conversations
        """)
    with col_b:
        st.markdown("""
**Translation Model**
- NLLB-200 600M (Meta)
- Runs locally on CPU
- Pashto → English, Urdu
- Verified by back-translation

**AI Assistant**
- HuggingFace free inference API
- Fallback: Pollinations.ai (no token)
- 3 strategies tried automatically
        """)
    st.markdown("---")
    st.subheader("Tips for Best Results")
    st.info("""
- 🎙️ **Audio quality** — minimal background noise gives best accuracy.
- ⏱️ **Clip length** — under 30 seconds works best.
- 📢 **Speaking pace** — clear, natural pace gives lowest WER.
- 🌍 **Dialect** — best results with Yusufzai/Peshawari (KPK) varieties.
- ✏️ **Edit and save** — corrected transcriptions are used for exports.
    """)


def auth_page():
    st.markdown(
        '<style>.auth-title{font-size:2.4rem;font-weight:800;'
        'background:linear-gradient(135deg,#6366f1,#a78bfa);'
        '-webkit-background-clip:text;-webkit-text-fill-color:transparent;margin-bottom:0.2rem;}'
        '.auth-sub{color:#94a3b8;font-size:1rem;margin-bottom:2rem;}</style>'
        '<div class="auth-title">Pashto Whisper</div>'
        '<div class="auth-sub">Merged LoRA Fine-Tuned Speech Transcription</div>',
        unsafe_allow_html=True
    )
    tab_login, tab_signup = st.tabs(["Login", "Create Account"])
    with tab_login:
        with st.form("login_form"):
            username = st.text_input("Username")
            password = st.text_input("Password", type="password")
            if st.form_submit_button("Login", use_container_width=True, type="primary"):
                ok, uid = verify_user(username.strip(), password)
                if ok:
                    st.session_state.logged_in = True
                    st.session_state.user_id   = uid
                    st.session_state.username  = username.strip()
                    st.rerun()
                else:
                    st.error("Invalid username or password.")
    with tab_signup:
        with st.form("signup_form"):
            new_user  = st.text_input("Username")
            new_email = st.text_input("Email (optional)")
            new_pass  = st.text_input("Password", type="password")
            confirm   = st.text_input("Confirm Password", type="password")
            if st.form_submit_button("Create Account", use_container_width=True, type="primary"):
                if len(new_user.strip()) < 3:
                    st.error("Username must be at least 3 characters.")
                elif len(new_pass) < 6:
                    st.error("Password must be at least 6 characters.")
                elif new_pass != confirm:
                    st.error("Passwords do not match.")
                else:
                    ok, msg = create_user(new_user.strip(), new_pass, new_email.strip())
                    st.success(msg) if ok else st.error(msg)


def transcribe_page():
    st.header("Transcribe Audio")
    st.caption("Supported: " + ", ".join("." + f for f in SUPPORTED_FORMATS))

    uploaded_files = st.file_uploader(
        "Upload audio files", type=SUPPORTED_FORMATS, accept_multiple_files=True
    )
    ref_text = st.text_area(
        "Reference text (optional) — paste correct Pashto text to calculate WER",
        height=70, placeholder="Paste correct Pashto text here for WER..."
    )
    if not uploaded_files:
        st.info("Upload audio files above to begin transcription.")
        return
    st.markdown(f"**{len(uploaded_files)} file(s) selected**")
    if not st.button("Transcribe All", type="primary", use_container_width=True):
        return

    progress_bar = st.progress(0, text="Starting...")
    all_results  = []

    for i, f in enumerate(uploaded_files):
        st.markdown("---")
        st.markdown(f"**[{i+1}/{len(uploaded_files)}] {f.name}**")
        tmp_path = f"/tmp/upload_{i}_{f.name}"
        with open(tmp_path, "wb") as out:
            out.write(f.getbuffer())
        user_dir = os.path.join(AUDIO_STORE, str(st.session_state.user_id))
        os.makedirs(user_dir, exist_ok=True)
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        saved_path = os.path.join(user_dir, f"{ts}_{f.name}")
        with open(saved_path, "wb") as out:
            out.write(f.getbuffer())
        st.audio(tmp_path)
        with st.spinner(f"Transcribing {f.name}..."):
            text = transcribe_audio(tmp_path)
        wer_score = compute_wer(ref_text, text) if ref_text.strip() else None
        tid = save_transcription(
            st.session_state.user_id, f.name, text,
            audio_filepath=saved_path,
            reference=ref_text.strip() or None,
            wer_score=wer_score
        )
        all_results.append((tid, f.name, text, wer_score))
        st.text_area(f"Pashto Transcription — {f.name}", value=text, height=100,
                     key=f"res_{i}_{tid}")
        if wer_score is not None:
            wer_badge(wer_score)
        elif ref_text.strip() and not JIWER_AVAILABLE:
            st.warning("Install jiwer to enable WER: pip install jiwer")
        try:
            os.remove(tmp_path)
        except OSError:
            pass
        progress_bar.progress((i + 1) / len(uploaded_files),
                               text=f"Completed {i+1}/{len(uploaded_files)}")

    st.markdown("---")
    st.success(f"All {len(uploaded_files)} file(s) transcribed and saved!")
    if all_results:
        headers = ["#", "Filename", "Transcription", "WER (%)", "Saved At"]
        rows    = [[idx, fname, text, ws or "", datetime.now().isoformat()]
                   for idx, (tid, fname, text, ws) in enumerate(all_results, 1)]
        col_csv, col_xlsx = st.columns(2)
        with col_csv:
            st.download_button("Download Session as CSV",
                               data=make_csv_bytes(rows, headers),
                               file_name="session_transcriptions.csv",
                               mime="text/csv; charset=utf-8")
        with col_xlsx:
            st.download_button("Download Session as Excel",
                               data=make_excel_bytes(rows, headers),
                               file_name="session_transcriptions.xlsx",
                               mime="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet")


def history_page():
    st.header("My Transcription History")
    rows = get_user_history(st.session_state.user_id)
    if not rows:
        st.info("No transcriptions yet. Head to Transcribe to get started.")
        return
    st.markdown(f"**{len(rows)} transcription(s) on record**")

    export_headers = ["ID", "Filename", "Original", "Edited", "Reference",
                      "WER (%)", "English", "Urdu", "Audio Path", "Date"]
    export_rows = [list(r) for r in rows]

    col_a, col_b, col_c = st.columns(3)
    with col_a:
        st.download_button("Export All as CSV",
                           data=make_csv_bytes(export_rows, export_headers),
                           file_name="all_transcriptions.csv",
                           mime="text/csv; charset=utf-8", use_container_width=True)
    with col_b:
        st.download_button("Export All as Excel",
                           data=make_excel_bytes(export_rows, export_headers),
                           file_name="all_transcriptions.xlsx",
                           mime="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
                           use_container_width=True)
    with col_c:
        lines = []
        for r in rows:
            tid, fname, orig, edited, ref, ws, en_t, ur_t, audio_fp, ts = r
            lines += ["="*60, f"File   : {fname}",
                      f"Date   : {ts[:16].replace('T',' ')}",
                      f"WER    : {str(ws)+'%' if ws is not None else 'N/A'}",
                      f"Pashto :\n{edited or orig or ''}"]
            if en_t: lines.append(f"English:\n{en_t}")
            if ur_t: lines.append(f"Urdu   :\n{ur_t}")
            lines.append("")
        st.download_button("Export All as TXT",
                           data="\n".join(lines).encode("utf-8"),
                           file_name="all_transcriptions.txt",
                           mime="text/plain; charset=utf-8", use_container_width=True)
    st.markdown("---")

    for r in rows:
        tid, filename, original, edited, reference, wer_score, en_t, ur_t, audio_filepath, created_at = r
        date_str = created_at[:16].replace("T", " ") if created_at else ""
        label = f"{filename}  |  {date_str}"
        if wer_score is not None:
            label += f"  |  WER {wer_score}%"
        with st.expander(label):
            st.caption(f"Record #{tid}")
            if audio_filepath and os.path.exists(audio_filepath):
                st.audio(audio_filepath, format="audio/*", start_time=0)
            else:
                st.warning(f"Audio file not found: {filename}")
            if original and original != (edited or original):
                with st.expander("View original (unedited)"):
                    st.text(original)
            edited_val = st.text_area("Pashto Transcription (editable)",
                                      value=edited or original or "",
                                      key=f"edit_{tid}", height=120)
            new_ref = st.text_input("Reference text for WER", value=reference or "",
                                    key=f"ref_{tid}", placeholder="Paste correct Pashto text...")
            if wer_score is not None:
                wer_badge(wer_score)
            col1, col2, col3, col4 = st.columns(4)
            with col1:
                if st.button("Save Edits", key=f"save_{tid}", use_container_width=True, type="primary"):
                    new_wer = compute_wer(new_ref, edited_val) if new_ref.strip() else None
                    update_transcription(tid, edited_val, new_ref.strip() or None, new_wer)
                    st.success("Saved!")
                    st.rerun()
            with col2:
                dk = f"confirm_delete_{tid}"
                if st.session_state.get(dk, False):
                    st.warning("Are you sure?")
                    if st.button("Confirm Delete", key=f"confirm_del_{tid}",
                                 use_container_width=True, type="primary"):
                        delete_transcription(tid, audio_filepath)
                        st.session_state[dk] = False
                        st.rerun()
                    if st.button("Cancel", key=f"cancel_del_{tid}", use_container_width=True):
                        st.session_state[dk] = False
                        st.rerun()
                else:
                    if st.button("Remove", key=f"remove_{tid}", use_container_width=True, type="primary"):
                        st.session_state[dk] = True
                        st.rerun()
            with col3:
                st.download_button("Download TXT", data=edited_val.encode("utf-8"),
                                   file_name=f"{filename}.txt", mime="text/plain; charset=utf-8",
                                   key=f"dl_t_{tid}", use_container_width=True)
            with col4:
                sh = ["Filename", "Pashto", "English", "Urdu", "WER (%)", "Date"]
                sr = [[filename, edited_val, en_t or "", ur_t or "", wer_score or "", created_at]]
                st.download_button("Download CSV", data=make_csv_bytes(sr, sh),
                                   file_name=f"{filename}.csv", mime="text/csv; charset=utf-8",
                                   key=f"dl_c_{tid}", use_container_width=True)
            st.markdown("---")
            show_translation_box(tid, edited_val, en_t, ur_t)


def ai_chat_page():
    st.header("AI Assistant")
    st.caption("Ask anything about your Pashto transcriptions — free AI, no paid subscription needed.")
    if not HF_TOKEN:
        st.warning(
            "**HF_TOKEN not set** — AI quality is better with a free token.\n\n"
            "Get one free at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)\n\n"
            "Set it in Colab: `import os; os.environ['HF_TOKEN'] = 'hf_...'` then restart.\n\n"
            "Without a token, Pollinations.ai fallback is used automatically."
        )
    rows = get_user_history(st.session_state.user_id)
    trans_labels = ["(none — type your own context)"]
    trans_texts  = [""]
    for r in rows:
        tid, fname, orig, edited, ref, ws, en_t, ur_t, audio_fp, ts = r
        trans_labels.append(f"{fname} | {ts[:10]}")
        trans_texts.append(edited or orig or "")
    idx = st.selectbox("Load a saved transcription as context",
                       range(len(trans_labels)), format_func=lambda i: trans_labels[i])
    context_text = st.text_area("Context", value=trans_texts[idx], height=130,
                                 placeholder="Load transcription above or paste any text here...")
    system_msg = st.text_input(
        "System instruction",
        value="You are a helpful assistant specializing in Pashto language and transcription analysis.",
        key="sys_msg"
    )
    user_question = st.text_area("Your question", height=100,
                                  placeholder="Examples:\n— Summarize in English\n— What topics are discussed?\n— Fix grammatical errors")
    if st.button("Ask AI", type="primary", use_container_width=True):
        if not user_question.strip():
            st.warning("Please type a question first.")
        else:
            prompt = ""
            if context_text.strip():
                prompt = "Transcription context:\n" + context_text.strip() + "\n\n"
            prompt += user_question.strip()
            with st.spinner("Asking AI (up to 60 seconds on free tier)..."):
                answer, err = ask_ai_free(prompt, system_msg)
            if err:
                st.error(err)
            else:
                st.markdown("---")
                st.markdown("**AI Response:**")
                st.markdown(answer)
                st.download_button(
                    "Download AI Response",
                    data=f"Question:\n{user_question}\n\nContext:\n{context_text}\n\nAnswer:\n{answer}".encode("utf-8"),
                    file_name="ai_response.txt", mime="text/plain; charset=utf-8"
                )

# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    st.set_page_config(page_title="Pashto Whisper", page_icon="🎙", layout="wide",
                       initial_sidebar_state="expanded")
    st.markdown(
        '<style>[data-testid="stSidebar"]{background:#0f172a;}'
        '[data-testid="stSidebar"] *{color:#e2e8f0 !important;}</style>',
        unsafe_allow_html=True
    )
    if "logged_in" not in st.session_state:
        st.session_state.logged_in = False
    if not st.session_state.logged_in:
        auth_page()
        return

    with st.sidebar:
        st.markdown("## 🎙️ Pashto Whisper")
        st.markdown(f"**User: {st.session_state.username}**")
        st.markdown("---")
        page = st.radio("Navigate", ["About", "Transcribe", "History", "AI Assistant"],
                        label_visibility="collapsed")
        st.markdown("---")
        if not JIWER_AVAILABLE:
            st.warning("WER disabled. Run: pip install jiwer")
        device_label = "GPU (CUDA)" if torch.cuda.is_available() else "CPU"
        st.caption(f"Device     : {device_label}")
        st.caption(f"FP16       : {'ON' if FP16_ENABLED else 'OFF'}")
        st.caption("Model      : Merged Whisper+LoRA")
        st.caption("AI         : HF Free + Pollinations")
        st.caption("Translation: NLLB-600M (local CPU)")
        st.markdown("---")
        if st.button("Logout", use_container_width=True):
            for k in ("logged_in", "user_id", "username"):
                st.session_state[k] = None
            st.session_state.logged_in = False
            st.rerun()

    if page == "About":
        about_page()
    elif page == "Transcribe":
        transcribe_page()
    elif page == "History":
        history_page()
    else:
        ai_chat_page()


main()

Overwriting app.py


### Run the Streamlit Application

To run the Streamlit app, we'll use `ngrok` to create a public URL that you can access from your browser. Follow these steps:

1.  **Get an `ngrok` Authtoken:** If you don't have one, sign up at [ngrok.com](https://ngrok.com/) and get your authtoken.
2.  **Paste your `ngrok` Authtoken:** Run the cell below, and replace `YOUR_NGROK_AUTH_TOKEN` with your actual token.
3.  **Run the Streamlit app:** Execute the second cell to launch the app. It will provide a public URL.
4.  **Open the URL:** Click on the provided `ngrok` URL to open your Streamlit app in a new tab.

In [ ]:
from pyngrok import ngrok

# Replace 'YOUR_NGROK_AUTH_TOKEN' with your actual ngrok authtoken
NGROK_AUTH_TOKEN = "Replace_with_your_actual_ngrok_auth_token" # You have already replaced this

# Authenticate ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print(" ngrok authenticated.")

 ngrok authenticated.


### Replace text with your Actual Hugging Face Token

In [ ]:
import subprocess
import threading
import time
import os
from pyngrok import ngrok # Import ngrok as it's used later in the cell

# Re-define necessary global variables for this cell's scope
OUTPUT_DIR   = "/content/drive/MyDrive/Audios_Phusto/whisper-pashto-lora"
MODEL_NAME   = "openai/whisper-small"
CACHE_DIR    = "/content/cache"
LORA_R       = 128
LORA_ALPHA   = 256
LORA_DROPOUT = 0.15
# FP16 should be True for GPU usage, aligning with notebook's configuration
FP16 = True
# Path to the fully merged model, as set in previous cells (e.g., YsiIzPRbZDYA)
MERGED_MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, "merged_whisper_lora_pashto")

def run_streamlit():
    # Set environment variables for the Streamlit app to access global notebook values
    os.environ['MODEL_NAME'] = MODEL_NAME
    os.environ['CACHE_DIR'] = CACHE_DIR
    os.environ['LORA_R'] = str(LORA_R)
    os.environ['LORA_ALPHA'] = str(LORA_ALPHA)
    os.environ['LORA_DROPOUT'] = str(LORA_DROPOUT)
    # Set MERGED_MODEL_PATH for the Streamlit app to load the fully merged model
    os.environ['MERGED_MODEL_PATH'] = MERGED_MODEL_SAVE_PATH
    os.environ['FP16'] = str(FP16) # Use the global FP16 value

    # Set the Hugging Face API token
    os.environ['HF_TOKEN'] = 'Replace with your actual HF TOKEN' # <<< IMPORTANT: REPLACE WITH YOUR ACTUAL HF TOKEN

    # Run Streamlit in a background thread using 'python -m streamlit'
    cmd = ["python", "-m", "streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "True"]
    process = subprocess.Popen(cmd)
    print("Streamlit process started.")
    return process

# Start Streamlit
streamlit_process = run_streamlit()

# Give Streamlit a moment to start up
time.sleep(5)

# Open ngrok tunnel
public_url = ngrok.connect(8501)
print(f" Your Streamlit app is live at: {public_url}")
print("To stop the app, interrupt this cell (Runtime > Interrupt execution).")

# Keep the cell alive to maintain the tunnel
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nStopping Streamlit and ngrok tunnel...")
    streamlit_process.terminate()
    ngrok.kill()
    print("App stopped.")

: 